# Analog Holidays - 38h Offset Forecast + Cluster Filter

Thin orchestration notebook for running `AnalogSpecialDays` from the hourly wide holiday audit CSV.

This variant forecasts a 38-hour window that starts 14 hours before the holiday midnight, so the operational forecast is ready before the holiday begins.

The loading, normalization, forecasting, and plotting logic lives in `analog/analog_holidays.py`. This notebook only defines parameters and calls the plotting helpers.

The CSV export only preserves holiday flags, so this workflow targets holiday analogs and restricts the analog bank to the selector cluster `F/G/H` assigned to each target in `holiday_selector_features.csv`.

In [1]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import analog_holidays.analog.analog_special_days as analog_special_days_module
import analog_holidays.analog.analog_holidays as analog_holidays_module

analog_special_days_module = importlib.reload(analog_special_days_module)
analog_holidays_module = importlib.reload(analog_holidays_module)

from analog_holidays.analog.analog_holidays import (
    build_analog_ranking_table,
    build_run_summary,
    plot_analog_pair_sequences,
    plot_batch_inference_grid,
    plot_batch_pair_sequences_grid,
    plot_forecast_diagnostics,
    plot_ranked_analog_profiles,
    prepare_audit_working_copy,
    run_analog_holidays,
    run_analog_holidays_batch,
    tune_analog_holidays_optuna,
)

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 20)

## Parameters

Adjust the series, target date, and analog hyperparameters for the holiday-only CSV source.

- SOURCE_PATH: path to the holiday CSV file consumed by the notebook.
- UNIQUE_ID: target series used to build analogs and generate the forecast.
- TARGET_DATE: holiday date whose operational forecast window should be estimated.
- FORECAST_START_OFFSET_HOURS: how many hours before the holiday midnight the forecast window starts.
- SEASON_LENGTH: hourly profile length to model, in hours. For this workflow it should match `FORECAST_START_OFFSET_HOURS + 24`.
- SPECIAL_LABELS: labels that define which days are treated as special candidates when selecting analogs.
- K: number of special neighbors kept after ranking X against Y by similarity; None uses every filtered candidate.
- TYPEDIST: metric used to rank holiday candidates against Y in a fixed manual run; supports pearson, euclidian, and dtw.
- TYPEREG: regressor type used in the analog reconstruction step for a fixed manual run; supports PCR, PLS, RidgeReg, LassoReg, RF, OLSstep, and LGBM when `lightgbm` is installed.
- SCALE_METHOD: optional preprocessing transform applied before neighbor selection and regression, then inverted back to the original demand scale for the final forecast. Supported values: `None`, `standard`, `minmax`.
- N_COMPONENTS: number of components for dimensionality-reduction methods such as PCR or PLS; ignored by tree-based regressors.
- REGRESSOR_PARAMS: optional dict of model-specific constructor hyperparameters, mainly for RF or LGBM when running a manual configuration outside Optuna.
- OPTUNA_TYPEDIST_CHOICES: explicit search grid for the Optuna `typedist` parameter.
- OPTUNA_TYPEREG_CHOICES: explicit search grid for the Optuna `typereg` parameter.
- OPTUNA_SCALE_METHOD_CHOICES: explicit search grid for the Optuna `scale_method` parameter.
- OPTUNA_MIN_K: lower bound of the `k` search range explored by Optuna during rolling tuning; the upper bound is resolved dynamically from the realizable analog pool.
- OPTUNA_N_TRIALS / OPTUNA_TIMEOUT_SEC / OPTUNA_MAX_EVAL_DATES / OPTUNA_RANDOM_SEED: Optuna budget, rolling evaluation coverage, and reproducibility controls.
- LEVELS: prediction interval levels to compute for the forecast.
- MIN_SPECIAL_POINTS: minimum number of hours flagged as special inside a candidate block.
- MIN_EVENT_GAP: minimum separation between consecutive special events to avoid overly overlapping candidates.
- MAX_EVENTS: maximum number of special events used as the analog bank; None uses all available events.
- SELECTOR_FEATURES_PATH / CLUSTER_COLUMN / USE_CLUSTER / MATCH_TARGET_CLUSTER: optional selector-cluster filter. When `USE_CLUSTER=False`, the analog algorithm chooses neighbors from the full historical holiday pool without restricting to `F/G/H`.
- MAX_PLOTTED_ANALOGS: maximum number of analogs shown in the comparative plots.

In [2]:
from datetime import datetime

# Original CSV ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â never modified by this notebook.
_ORIGINAL_SOURCE = PROJECT_ROOT / 'analog_holidays' / 'holidays' / 'holiday_demand_ercot.csv'
_WORKING_SOURCE_DIR = _ORIGINAL_SOURCE.parent / 'working'

def _ensure_working_source_path(source_path=SOURCE_PATH if 'SOURCE_PATH' in globals() else None):
    candidate = None if source_path is None else Path(source_path)
    if candidate is not None and candidate.exists():
        return candidate
    refreshed = prepare_audit_working_copy(
        _ORIGINAL_SOURCE,
        working_dir=_WORKING_SOURCE_DIR,
        prefix='holiday_demand_ercot',
        reuse_today=True,
    )
    print(f'Working copy refreshed: {refreshed.name}')
    return refreshed

# Create or reuse a working copy for this notebook session.
SOURCE_PATH = _ensure_working_source_path()

# Ablation knob: drop early history (e.g. pandemic years) from the analog pool by trimming
# the working source to ds >= HISTORY_START. None = use all available history (2020+).
HISTORY_START = None  # e.g. '2023-01-01' to exclude 2020-2021 COVID analogs
if HISTORY_START is not None:
    _src_full = pd.read_csv(SOURCE_PATH, parse_dates=['ds'])
    _src_trim = _src_full[_src_full['ds'] >= pd.Timestamp(HISTORY_START)].reset_index(drop=True)
    _trim_path = Path(SOURCE_PATH).with_name(Path(SOURCE_PATH).stem + f"_from{HISTORY_START.replace('-','')}.csv")
    _src_trim.to_csv(_trim_path, index=False)
    SOURCE_PATH = _trim_path
    print(f'History trimmed to >= {HISTORY_START}: {len(_src_trim)} of {len(_src_full)} rows -> {Path(SOURCE_PATH).name}')
print(f'Working copy: {Path(SOURCE_PATH).name}')

UNIQUE_IDS = analog_holidays_module.get_available_unique_ids(SOURCE_PATH)
if not UNIQUE_IDS:
    raise ValueError(f'No unique_id values were found in {Path(SOURCE_PATH).name}.')

UNIQUE_ID = 'ERCOT_demand_ERCOT'
if UNIQUE_ID not in UNIQUE_IDS:
    UNIQUE_ID = UNIQUE_IDS[0]

print(f'Series to forecast: {len(UNIQUE_IDS)}')
print(', '.join(UNIQUE_IDS))
print(f'Detail view series: {UNIQUE_ID}')

Working copy refreshed: holiday_demand_ercot_20260625_110105.csv
Working copy: holiday_demand_ercot_20260625_110105.csv
Series to forecast: 9
ERCOT_demand_COAST, ERCOT_demand_EAST, ERCOT_demand_ERCOT, ERCOT_demand_FWEST, ERCOT_demand_NCENT, ERCOT_demand_NORTH, ERCOT_demand_SCENT, ERCOT_demand_SOUTH, ERCOT_demand_WEST
Detail view series: ERCOT_demand_ERCOT


In [3]:
SPECIAL_LABELS = ('holiday',)
FORECAST_START_OFFSET_HOURS = 14
SEASON_LENGTH = 38  # 14 h pre-holiday + 24 h holiday
K = None  # No limit on analog neighbors (use all ranked candidates)
OPTUNA_MIN_K = 2
# Per-tier neighbour floor (observance_tier clusters): fully-observed holidays (H=full)
# tune better with more analogs; soft holidays (F=working, G=partial) prefer k=2.
# Experiments 2026_06_15 showed the civic/full gain comes from a higher MIN_K, not extra tiers.
OPTUNA_MIN_K_BY_CLUSTER = {'F': 2, 'G': 2, 'H': 2}
# Per-cluster k CEILING (mirror of MIN_K): cap cluster H (full observance) at k<=6;
# deep+civic H cells over-select k and blow up at k>=7 (BITACORA finding 8). None = no cap.
OPTUNA_MAX_K_BY_CLUSTER = {}
TYPEDIST = 'pearson'
TYPEREG = 'PCR'
SCALE_METHOD = None
OPTUNA_TYPEDIST_CHOICES = ['pearson', 'euclidian']
OPTUNA_TYPEREG_CHOICES = ['PCR', 'PLS','RidgeReg', 'LassoReg',]
OPTUNA_SCALE_METHOD_CHOICES = [None, 'standard', 'minmax']
N_COMPONENTS = 2 #
REGRESSOR_PARAMS = {}
LEVELS = [50, 80, 95]
MIN_SPECIAL_POINTS = 24  # require the 24 holiday hours inside each 38-h candidate window
MIN_EVENT_GAP = 24  # 24h between candidate events; gap=12 was worse on deep holidays (BITACORA finding 7)
MAX_EVENTS = None  # No limit on historical candidate events
MAX_PLOTTED_ANALOGS = 50
RECENT_WEEKEND_ANALOGS = 0

SELECTOR_FEATURES_PATH = PROJECT_ROOT / 'analog_holidays' / 'holidays' / 'holiday_selector_features_ercot.csv'
CLUSTER_COLUMN = 'analog_cluster'
USE_CLUSTER = True
MATCH_TARGET_CLUSTER = bool(USE_CLUSTER)

# Identify this run: per-holiday-identity hard filter (pure Similar-Days) experiment.
EXPERIMENT_SLUG = 'holiday_identity_cluster_ercot'

OPTUNA_N_TRIALS = 25
OPTUNA_TIMEOUT_SEC = 300
OPTUNA_MAX_EVAL_DATES = 19  # Tune across all 19 target dates for robust hyperparameters
OPTUNA_RANDOM_SEED = 42

HOURLY_FACTOR_ANALOGS = 2


In [4]:
if OPTUNA_TYPEDIST_CHOICES is not None:
    valid_typedist_choices = {'pearson', 'euclidian', 'dtw'}
    invalid_typedist_choices = [
        value for value in OPTUNA_TYPEDIST_CHOICES
        if value not in valid_typedist_choices
    ]
    if invalid_typedist_choices:
        raise ValueError(
            f'Unsupported OPTUNA_TYPEDIST_CHOICES: {invalid_typedist_choices}'
        )

if OPTUNA_TYPEREG_CHOICES is not None:
    valid_typereg_choices = {'PCR', 'PLS', 'RidgeReg', 'LassoReg', 'RF', 'OLSstep', 'LGBM'}
    invalid_typereg_choices = [
        value for value in OPTUNA_TYPEREG_CHOICES
        if value not in valid_typereg_choices
    ]
    if invalid_typereg_choices:
        raise ValueError(
            f'Unsupported OPTUNA_TYPEREG_CHOICES: {invalid_typereg_choices}'
        )

if OPTUNA_SCALE_METHOD_CHOICES is not None:
    valid_scale_methods = {None, 'standard', 'minmax'}
    invalid_scale_methods = [
        value for value in OPTUNA_SCALE_METHOD_CHOICES
        if value not in valid_scale_methods
    ]
    if invalid_scale_methods:
        raise ValueError(
            f'Unsupported OPTUNA_SCALE_METHOD_CHOICES: {invalid_scale_methods}'
        )

if int(OPTUNA_MIN_K) < 1:
    raise ValueError(f'OPTUNA_MIN_K must be >= 1, got {OPTUNA_MIN_K}.')

print(f'USE_CLUSTER: {USE_CLUSTER}')
print(f'MATCH_TARGET_CLUSTER: {MATCH_TARGET_CLUSTER}')
print(f'OPTUNA typedist choices: {OPTUNA_TYPEDIST_CHOICES}')
print(f'OPTUNA typereg choices: {OPTUNA_TYPEREG_CHOICES}')
print(f'OPTUNA_MIN_K: {OPTUNA_MIN_K}')
print(f'SCALE_METHOD fixed fallback: {SCALE_METHOD}')
print(f'OPTUNA scale choices: {OPTUNA_SCALE_METHOD_CHOICES}')

USE_CLUSTER: True
MATCH_TARGET_CLUSTER: True
OPTUNA typedist choices: ['pearson', 'euclidian']
OPTUNA typereg choices: ['PCR', 'PLS', 'RidgeReg', 'LassoReg']
OPTUNA_MIN_K: 2
SCALE_METHOD fixed fallback: None
OPTUNA scale choices: [None, 'standard', 'minmax']


In [5]:
TARGET_DATES_2025 = [
    ('2025-01-01', "New Year's Day"),
    ('2025-01-20', 'Martin Luther King Jr. Day'),
    ('2025-02-17', "Presidents' Day"),
    ('2025-03-02', 'Texas Independence Day'),
    ('2025-04-21', 'San Jacinto Day'),
    ('2025-05-26', 'Memorial Day'),
    ('2025-06-19', 'Juneteenth National Independence Day'),
    ('2025-07-04', 'Independence Day'),
    ('2025-08-27', 'Lyndon B. Johnson Day'),
    ('2025-09-01', 'Labor Day'),
    ('2025-11-11', 'Veterans Day'),
    ('2025-11-27', 'Thanksgiving Day'),
    ('2025-11-28', 'Day after Thanksgiving'),
    ('2025-12-24', 'Christmas Eve'),
    ('2025-12-25', 'Christmas Day'),
    # 2026 targets present in holiday_demand_ercot.csv / selector features.
    ('2026-01-01', "New Year's Day"),
    ('2026-01-19', 'Martin Luther King Jr. Day'),
    ('2026-02-16', "Presidents' Day"),
    ('2026-03-02', 'Texas Independence Day'),
]

TARGET_DATE = TARGET_DATES_2025[-1][0]

from matplotlib.backends.backend_pdf import PdfPages

RESULTS_DIR = PROJECT_ROOT / 'analog_holidays' / 'Holiday_results_ercot'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MULTIPAGE_PDF_PATH = RESULTS_DIR / 'all_holiday_graphs_ercot.pdf'
PDF_FIGURES = {}

def _pdf_safe_name(value):
    safe = ''.join(ch if ch.isalnum() or ch in ('-', '_') else '_' for ch in str(value))
    while '__' in safe:
        safe = safe.replace('__', '_')
    return safe.strip('_') or 'figure'

def export_figure_pdf(fig_obj, stem):
    safe_stem = _pdf_safe_name(stem)
    single_path = RESULTS_DIR / f'{safe_stem}.pdf'
    fig_obj.savefig(single_path, format='pdf', bbox_inches='tight')
    PDF_FIGURES[safe_stem] = fig_obj
    with PdfPages(MULTIPAGE_PDF_PATH) as pdf:
        for saved_fig in PDF_FIGURES.values():
            pdf.savefig(saved_fig, bbox_inches='tight')
    print(f'Saved PDF: {single_path}')
    print(f'Updated multipage PDF: {MULTIPAGE_PDF_PATH}')
    return single_path, MULTIPAGE_PDF_PATH

print(f'PDF output folder: {RESULTS_DIR}')


PDF output folder: /home/uriel/GIT/analog_holidays/Holiday_results_ercot


In [6]:
selector_features_df = pd.read_csv(SELECTOR_FEATURES_PATH, parse_dates=['date'])
selector_features_df['date'] = pd.to_datetime(selector_features_df['date']).dt.normalize()
if 'unique_id' not in selector_features_df.columns:
    raise ValueError(
        f'{SELECTOR_FEATURES_PATH.name} must contain a unique_id column. '
        'Re-export the selector from M_identify_holidays.ipynb.'
    )

selector_features_current_df = selector_features_df.loc[
    selector_features_df['unique_id'].astype(str) == str(UNIQUE_ID)
] .copy()
if selector_features_current_df.empty:
    raise ValueError(
        f'No selector rows were found for UNIQUE_ID={UNIQUE_ID!r} in {SELECTOR_FEATURES_PATH.name}.'
    )

_cluster_filter_values = []
if 'analog_cluster_criterion' in selector_features_current_df.columns:
    _cluster_filter_values = (
        selector_features_current_df['analog_cluster_criterion']
        .dropna()
        .astype(str)
        .map(str.strip)
        .unique()
        .tolist()
    )
    _cluster_filter_values = [value for value in _cluster_filter_values if value]
if len(_cluster_filter_values) > 1:
    raise ValueError(
        f'{SELECTOR_FEATURES_PATH.name} contains multiple analog_cluster_criterion values for UNIQUE_ID={UNIQUE_ID!r}: '
        f'{_cluster_filter_values}'
    )

CLUSTER_FILTER_LABEL = (
    _cluster_filter_values[0]
    if MATCH_TARGET_CLUSTER and _cluster_filter_values
    else (True if MATCH_TARGET_CLUSTER else False)
)

target_ts = pd.Timestamp(TARGET_DATE).normalize()

target_cluster_cols = ['unique_id', 'date', 'holiday_name', 'holiday_day_type', CLUSTER_COLUMN]
if 'analog_cluster_criterion' in selector_features_current_df.columns:
    target_cluster_cols.append('analog_cluster_criterion')

target_cluster_df = selector_features_current_df.loc[
    selector_features_current_df['date'] == target_ts,
    target_cluster_cols,
]

if MATCH_TARGET_CLUSTER:
    if target_cluster_df.empty:
        raise ValueError(
            f'No selector cluster was found for UNIQUE_ID={UNIQUE_ID!r} and TARGET_DATE={target_ts.date()} '
            f'in {SELECTOR_FEATURES_PATH.name}.'
        )
    TARGET_ANALOG_CLUSTER = target_cluster_df.iloc[0][CLUSTER_COLUMN]
    eligible_cluster_analogs_df = selector_features_current_df.loc[
        (selector_features_current_df[CLUSTER_COLUMN] == TARGET_ANALOG_CLUSTER)
        & (selector_features_current_df['date'] < target_ts),
        ['unique_id', 'date', 'holiday_name', 'anchor_holiday_name', 'holiday_day_type', CLUSTER_COLUMN],
    ].sort_values('date').reset_index(drop=True)
    print(
        f'UNIQUE_ID={UNIQUE_ID} | TARGET_DATE={target_ts.date()} -> analog_cluster={TARGET_ANALOG_CLUSTER} '
        f'| cluster_filter={CLUSTER_FILTER_LABEL} '
        f'| eligible historical analog dates={len(eligible_cluster_analogs_df)}'
    )
    display(target_cluster_df)
else:
    TARGET_ANALOG_CLUSTER = pd.NA
    eligible_cluster_analogs_df = selector_features_current_df.loc[
        selector_features_current_df['date'] < target_ts,
        ['unique_id', 'date', 'holiday_name', 'anchor_holiday_name', 'holiday_day_type', CLUSTER_COLUMN],
    ].sort_values('date').reset_index(drop=True)
    print(
        f'UNIQUE_ID={UNIQUE_ID} | TARGET_DATE={target_ts.date()} -> cluster_filter={CLUSTER_FILTER_LABEL} '
        f'| historical holiday rows={len(eligible_cluster_analogs_df)}'
    )
    if not target_cluster_df.empty:
        display(target_cluster_df)

display(eligible_cluster_analogs_df)

UNIQUE_ID=ERCOT_demand_ERCOT | TARGET_DATE=2026-03-02 -> analog_cluster=R | cluster_filter=holiday_identity | eligible historical analog dates=9


,unique_id,date,holiday_name,holiday_day_type,analog_cluster,analog_cluster_criterion
467,ERCOT_demand_ERCOT,2026-03-02,Texas Independence Day,H2,R,holiday_identity


,unique_id,date,holiday_name,anchor_holiday_name,holiday_day_type,analog_cluster
0,ERCOT_demand_ERCOT,2016-03-02,Texas Independence Day,Texas Independence Day,H2,R
1,ERCOT_demand_ERCOT,2017-03-02,Texas Independence Day,Texas Independence Day,H2,R
2,ERCOT_demand_ERCOT,2018-03-02,Texas Independence Day,Texas Independence Day,H2,R
3,ERCOT_demand_ERCOT,2019-03-02,Texas Independence Day,Texas Independence Day,H2,R
4,ERCOT_demand_ERCOT,2020-03-02,Texas Independence Day,Texas Independence Day,H2,R
5,ERCOT_demand_ERCOT,2022-03-02,Texas Independence Day,Texas Independence Day,H2,R
6,ERCOT_demand_ERCOT,2023-03-02,Texas Independence Day,Texas Independence Day,H2,R
7,ERCOT_demand_ERCOT,2024-03-02,Texas Independence Day,Texas Independence Day,H2,R
8,ERCOT_demand_ERCOT,2025-03-02,Texas Independence Day,Texas Independence Day,H2,R


In [7]:
SOURCE_PATH = _ensure_working_source_path(SOURCE_PATH)

rolling_target_items = [
    (pd.Timestamp(target_date).date().isoformat(), holiday_label)
    for target_date, holiday_label in TARGET_DATES_2025
]

series_unique_ids = list(UNIQUE_IDS) if 'UNIQUE_IDS' in globals() else [UNIQUE_ID]
if not series_unique_ids:
    raise ValueError('UNIQUE_IDS is empty.')
if UNIQUE_ID not in series_unique_ids:
    UNIQUE_ID = series_unique_ids[0]
series_unique_ids = [str(value) for value in series_unique_ids]
selector_lookup_df = selector_features_df.copy()
selector_lookup_df['unique_id'] = selector_lookup_df['unique_id'].astype(str)
cluster_filter_label = globals().get(
    'CLUSTER_FILTER_LABEL',
    True if MATCH_TARGET_CLUSTER else False,
)

selector_cluster_lookup_by_id = {
    series_unique_id: (
        selector_lookup_df.loc[selector_lookup_df['unique_id'] == series_unique_id]
        .dropna(subset=[CLUSTER_COLUMN])
        .drop_duplicates(subset=['date'], keep='last')
        .set_index('date')[CLUSTER_COLUMN]
        .to_dict()
    )
    for series_unique_id in series_unique_ids
}

selector_anchor_lookup_by_id = {
    series_unique_id: (
        selector_lookup_df.loc[selector_lookup_df['unique_id'] == series_unique_id]
        .dropna(subset=['anchor_holiday_name'])
        .drop_duplicates(subset=['date'], keep='last')
        .set_index('date')['anchor_holiday_name']
        .to_dict()
    )
    for series_unique_id in series_unique_ids
}

def _summary_param(summary_df, param_name, default=np.nan):
    matches = summary_df.loc[summary_df['param'] == param_name, 'value']
    return matches.iloc[0] if not matches.empty else default

rolling_optuna_results = {}
rolling_runs = {}
rolling_rows = []

for unique_id in series_unique_ids:
    print(f'=== UNIQUE_ID={unique_id} ===')
    series_cluster_lookup = selector_cluster_lookup_by_id.get(str(unique_id), {})
    series_anchor_lookup = selector_anchor_lookup_by_id.get(str(unique_id), {})
    series_optuna_results = {}
    series_runs = {}

    for target_date, holiday_label in rolling_target_items:
        target_ts = pd.Timestamp(target_date).normalize()
        target_cluster = series_cluster_lookup.get(target_ts, pd.NA)
        anchor_holiday_name = series_anchor_lookup.get(target_ts, holiday_label)
        if pd.isna(anchor_holiday_name):
            anchor_holiday_name = holiday_label
        anchor_holiday_name = str(anchor_holiday_name)
        _target_min_k = OPTUNA_MIN_K_BY_CLUSTER.get(str(target_cluster), OPTUNA_MIN_K) \
            if 'OPTUNA_MIN_K_BY_CLUSTER' in globals() else OPTUNA_MIN_K
        _target_max_k = OPTUNA_MAX_K_BY_CLUSTER.get(str(target_cluster), None) if 'OPTUNA_MAX_K_BY_CLUSTER' in globals() else None
        print(f'[{unique_id}] [{target_date}] tuning and forecasting [{anchor_holiday_name}]...')

        try:
            tuning_result = tune_analog_holidays_optuna(
                unique_id=unique_id,
                source_path=SOURCE_PATH,
                train_end=target_ts,
                season_length=SEASON_LENGTH,
                forecast_start_offset_hours=FORECAST_START_OFFSET_HOURS,
                initial_k=K,
                initial_typedist=TYPEDIST,
                initial_typereg=TYPEREG,
                typedist_choices=OPTUNA_TYPEDIST_CHOICES,
                typereg_choices=OPTUNA_TYPEREG_CHOICES,
                scale_method=SCALE_METHOD,
                scale_method_choices=OPTUNA_SCALE_METHOD_CHOICES,
                initial_n_components=N_COMPONENTS,
                initial_regressor_params=REGRESSOR_PARAMS,
                optuna_min_k=_target_min_k,
                optuna_max_k=_target_max_k,
                n_trials=OPTUNA_N_TRIALS,
                timeout_sec=OPTUNA_TIMEOUT_SEC,
                max_eval_dates=OPTUNA_MAX_EVAL_DATES,
                random_seed=OPTUNA_RANDOM_SEED,
                special_labels=SPECIAL_LABELS,
                min_special_points=MIN_SPECIAL_POINTS,
                min_event_gap=MIN_EVENT_GAP,
                max_events=MAX_EVENTS,
                selector_features_path=SELECTOR_FEATURES_PATH,
                cluster_column=CLUSTER_COLUMN,
                match_target_cluster=MATCH_TARGET_CLUSTER,
                recent_weekend_analogs=RECENT_WEEKEND_ANALOGS,
            )
            series_optuna_results[target_date] = tuning_result

            best_config = tuning_result.best_config
            best_k_range = tuple(best_config.get('k_range', (np.nan, np.nan)))
            best_scale_method = best_config.get('scale_method', SCALE_METHOD)
            best_regressor_params = dict(best_config.get('regressor_params', {}))
            run = run_analog_holidays(
                unique_id=unique_id,
                target_date=target_ts,
                source_path=SOURCE_PATH,
                season_length=SEASON_LENGTH,
                forecast_start_offset_hours=FORECAST_START_OFFSET_HOURS,
                k=int(best_config['k']),
                typedist=str(best_config['typedist']),
                typereg=str(best_config['typereg']),
                scale_method=best_scale_method,
                n_components=int(best_config['n_components']),
                regressor_params=best_regressor_params,
                levels=LEVELS,
                special_labels=SPECIAL_LABELS,
                min_special_points=MIN_SPECIAL_POINTS,
                min_event_gap=MIN_EVENT_GAP,
                max_events=MAX_EVENTS,
                expected_target_label=None,
                selector_features_path=SELECTOR_FEATURES_PATH,
                cluster_column=CLUSTER_COLUMN,
                match_target_cluster=MATCH_TARGET_CLUSTER,
                recent_weekend_analogs=RECENT_WEEKEND_ANALOGS,
            )
            series_runs[target_date] = run

            mae_window = np.nan
            mape_window_pct = np.nan
            if run.actual_profile is not None:
                mae_window = float(np.mean(np.abs(run.forecast_profile - run.actual_profile)))
                denom = np.where(np.abs(run.actual_profile) > 1e-9, np.abs(run.actual_profile), np.nan)
                ape_pct = np.abs(run.forecast_profile - run.actual_profile) / denom * 100.0
                if np.isfinite(ape_pct).any():
                    mape_window_pct = float(np.nanmean(ape_pct))

            rolling_rows.append({
                'unique_id': unique_id,
                'target_date': target_date,
                'holiday_label': holiday_label,
                'analog_cluster': target_cluster,
                'cluster_filter_label': cluster_filter_label,
                'filter_by_cluster': bool(MATCH_TARGET_CLUSTER),
                'train_end': target_date,
                'eligible_tuning_dates': len(tuning_result.eligible_dates),
                'target_exists': run.target_exists,
                'target_has_complete_profile': run.target_has_complete_profile,
                'selected_analogs': len(run.positions),
                'fail': run.fail,
                'optuna_k_min': best_k_range[0],
                'optuna_k_max': best_k_range[1],
                'k': run.k,
                'typedist': run.typedist,
                'typereg': run.typereg,
                'scale_method': run.scale_method,
                'n_components': run.n_components,
                'regressor_params': best_regressor_params,
                'forecast_start': run.forecast_start,
                'forecast_end': run.forecast_end,
                'mae_window': mae_window,
                'mape_window_pct': mape_window_pct,
                'tuning_best_mean_mae': _summary_param(tuning_result.summary_df, 'best_mean_mae'),
                'tuning_best_mean_mape_pct': _summary_param(tuning_result.summary_df, 'best_mean_mape_pct'),
                'error': None,
            })
            print(
                f'[{unique_id}] [{target_date}] cluster={cluster_filter_label} | '
                f'analog_cluster={target_cluster} | '
                f'k-range(optuna-param)=[{best_k_range[0]}, {best_k_range[1]}] | '
                f'k(optuna)={run.k} | '
                f'typereg(optuna)={run.typereg} | '
                f'typedist(optuna)={run.typedist} | '
                f'scale_method(optuna)={run.scale_method} | '
                f'MAPE(final)={mape_window_pct:.2f}%'
            )
        except Exception as exc:
            rolling_rows.append({
                'unique_id': unique_id,
                'target_date': target_date,
                'holiday_label': holiday_label,
                'analog_cluster': target_cluster,
                'cluster_filter_label': cluster_filter_label,
                'filter_by_cluster': bool(MATCH_TARGET_CLUSTER),
                'train_end': target_date,
                'eligible_tuning_dates': np.nan,
                'target_exists': False,
                'target_has_complete_profile': False,
                'selected_analogs': 0,
                'fail': True,
                'optuna_k_min': np.nan,
                'optuna_k_max': np.nan,
                'k': np.nan,
                'typedist': pd.NA,
                'typereg': pd.NA,
                'scale_method': pd.NA,
                'n_components': np.nan,
                'regressor_params': {},
                'forecast_start': pd.NaT,
                'forecast_end': pd.NaT,
                'mae_window': np.nan,
                'mape_window_pct': np.nan,
                'tuning_best_mean_mae': np.nan,
                'tuning_best_mean_mape_pct': np.nan,
                'error': str(exc),
            })
            print(f'[{unique_id}] [{target_date}] ERROR: {exc}')

    rolling_optuna_results[unique_id] = series_optuna_results
    rolling_runs[unique_id] = series_runs

rolling_daily_table = pd.DataFrame(rolling_rows)

batch_result_2025_all = {}
if not rolling_daily_table.empty:
    for series_unique_id, series_run_map in rolling_runs.items():
        series_results_df = (
            rolling_daily_table
            .loc[rolling_daily_table['unique_id'] == series_unique_id]
            .reset_index(drop=True)
        )
        if series_results_df.empty:
            continue
        metric_summary_columns = [
            column for column in [
                'mae_holiday24_bias_adjusted',
                'mape_holiday24_bias_adjusted_pct',
                'mae_holiday24_raw',
                'mape_holiday24_raw_pct',
                'mae_window_bias_adjusted',
                'mape_window_bias_adjusted_pct',
                'mae_window',
                'mape_window_pct',
            ]
            if column in series_results_df.columns
        ]
        batch_result_2025_all[series_unique_id] = analog_holidays_module.AnalogHolidayBatchResult(
            target_items=rolling_target_items,
            runs=series_run_map,
            results_df=series_results_df,
            metric_summary_df=series_results_df[metric_summary_columns].describe(include='all'),
        )

batch_result_2025 = batch_result_2025_all.get(UNIQUE_ID)
rolling_daily_table

=== UNIQUE_ID=ERCOT_demand_COAST ===
[ERCOT_demand_COAST] [2025-01-01] tuning and forecasting [New Year's Day]...
[ERCOT_demand_COAST] [2025-01-01] cluster=holiday_identity | analog_cluster=O | k-range(optuna-param)=[2, 6] | k(optuna)=4 | typereg(optuna)=RidgeReg | typedist(optuna)=pearson | scale_method(optuna)=minmax | MAPE(final)=8.16%
[ERCOT_demand_COAST] [2025-01-20] tuning and forecasting [Martin Luther King Jr. Day]...
[ERCOT_demand_COAST] [2025-01-20] cluster=holiday_identity | analog_cluster=M | k-range(optuna-param)=[2, 6] | k(optuna)=5 | typereg(optuna)=RidgeReg | typedist(optuna)=pearson | scale_method(optuna)=minmax | MAPE(final)=23.14%
[ERCOT_demand_COAST] [2025-02-17] tuning and forecasting [Presidents' Day]...
[ERCOT_demand_COAST] [2025-02-17] cluster=holiday_identity | analog_cluster=P | k-range(optuna-param)=[2, 6] | k(optuna)=4 | typereg(optuna)=PLS | typedist(optuna)=pearson | scale_method(optuna)=minmax | MAPE(final)=4.93%
[ERCOT_demand_COAST] [2025-03-02] tuning a

,unique_id,target_date,holiday_label,analog_cluster,cluster_filter_label,filter_by_cluster,train_end,eligible_tuning_dates,target_exists,target_has_complete_profile,...,scale_method,n_components,regressor_params,forecast_start,forecast_end,mae_window,mape_window_pct,tuning_best_mean_mae,tuning_best_mean_mape_pct,error
0,ERCOT_demand_COAST,2025-01-01,New Year's Day,O,holiday_identity,True,2025-01-01,19,True,True,...,minmax,2,{},2024-12-31 10:00:00,2025-01-02,914.398791,8.162703,890.616142,6.566,None
1,ERCOT_demand_COAST,2025-01-20,Martin Luther King Jr. Day,M,holiday_identity,True,2025-01-20,19,True,True,...,minmax,2,{},2025-01-19 10:00:00,2025-01-21,3761.647006,23.137946,902.231416,6.694,None
2,ERCOT_demand_COAST,2025-02-17,Presidents' Day,P,holiday_identity,True,2025-02-17,19,True,True,...,minmax,2,{},2025-02-16 10:00:00,2025-02-18,629.266696,4.929085,1075.012511,7.665,None
3,ERCOT_demand_COAST,2025-03-02,Texas Independence Day,R,holiday_identity,True,2025-03-02,19,True,True,...,minmax,2,{},2025-03-01 10:00:00,2025-03-03,451.624848,3.867997,1091.734632,7.779,None
4,ERCOT_demand_COAST,2025-04-21,San Jacinto Day,Q,holiday_identity,True,2025-04-21,19,True,True,...,minmax,2,{},2025-04-20 10:00:00,2025-04-22,778.548574,5.111901,1100.058853,7.844,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,ERCOT_demand_WEST,2025-12-25,Christmas Day,F,holiday_identity,True,2025-12-25,19,True,True,...,minmax,2,{},2025-12-24 10:00:00,2025-12-26,43.360475,3.306576,71.397132,5.301,None
167,ERCOT_demand_WEST,2026-01-01,New Year's Day,O,holiday_identity,True,2026-01-01,19,True,True,...,minmax,2,{},2025-12-31 10:00:00,2026-01-02,72.112089,5.420328,69.085842,5.089,None
168,ERCOT_demand_WEST,2026-01-19,Martin Luther King Jr. Day,M,holiday_identity,True,2026-01-19,19,True,True,...,minmax,2,{},2026-01-18 10:00:00,2026-01-20,152.503018,9.818527,69.880422,5.125,None
169,ERCOT_demand_WEST,2026-02-16,Presidents' Day,P,holiday_identity,True,2026-02-16,19,True,True,...,minmax,2,{},2026-02-15 10:00:00,2026-02-17,86.189761,6.212906,73.666790,5.299,None


In [8]:
# Ajuste dinámico 38h: estima la forma intradiaria sobre la ventana completa usando hasta los 4 análogos más similares disponibles.
HOURLY_FACTOR_ANALOGS = 4
BIAS_HEAD_HOURS = int(FORECAST_START_OFFSET_HOURS)
BIAS_TAIL_HOURS = int(SEASON_LENGTH - BIAS_HEAD_HOURS)

if BIAS_HEAD_HOURS <= 0 or BIAS_TAIL_HOURS <= 0:
    raise ValueError(
        f'Hourly holiday adjustment requires a valid 14h/24h split inside the 38h window. '
        f'Got FORECAST_START_OFFSET_HOURS={FORECAST_START_OFFSET_HOURS} and SEASON_LENGTH={SEASON_LENGTH}.'
    )
if 'rolling_daily_table' not in globals() or rolling_daily_table.empty:
    raise ValueError('Run the rolling study cell first to build rolling_daily_table.')
if 'rolling_runs' not in globals() or not rolling_runs:
    raise ValueError('Run the rolling study cell first to build rolling_runs.')

def _mean_abs_error(actual, forecast):
    return float(np.mean(np.abs(actual - forecast)))

def _mean_ape_pct(actual, forecast):
    denom = np.where(np.abs(actual) > 1e-9, np.abs(actual), np.nan)
    ape_pct = np.abs(actual - forecast) / denom * 100.0
    return float(np.nanmean(ape_pct)) if np.isfinite(ape_pct).any() else np.nan

def _mean_pct_error(actual, forecast):
    denom = np.where(np.abs(actual) > 1e-9, np.abs(actual), np.nan)
    pe_pct = (actual - forecast) / denom * 100.0
    return float(np.nanmean(pe_pct)) if np.isfinite(pe_pct).any() else np.nan

def _mean_bias(actual, forecast):
    return float(np.mean(actual - forecast))

def _build_window_metrics(actual, forecast, label):
    return {
        f'mae_{label}': _mean_abs_error(actual, forecast),
        f'mape_{label}_pct': _mean_ape_pct(actual, forecast),
        f'mpe_{label}_pct': _mean_pct_error(actual, forecast),
        f'bias_{label}': _mean_bias(actual, forecast),
    }

def _safe_corr(series_a, series_b):
    clean_df = pd.DataFrame({'a': series_a, 'b': series_b}).dropna()
    if len(clean_df) < 2 or clean_df['a'].nunique() < 2 or clean_df['b'].nunique() < 2:
        return np.nan
    return float(clean_df['a'].corr(clean_df['b']))

def _apply_hourly_factor_model(forecast_profile, factor_model, expected_hours):
    forecast_profile = np.asarray(forecast_profile, dtype=np.float64)
    resolved_expected_hours = int(expected_hours)
    if forecast_profile.shape[0] != resolved_expected_hours:
        raise ValueError(
            f'Expected a {resolved_expected_hours}-hour forecast, got shape {forecast_profile.shape}.'
        )
    model_window_hours = int(factor_model.get('window_hours', resolved_expected_hours))
    if model_window_hours != resolved_expected_hours:
        raise ValueError(
            f'Bias factor model window_hours={model_window_hours} does not match expected_hours={resolved_expected_hours}.'
        )
    forecast_window_mean = float(np.mean(forecast_profile))
    if factor_model['train_samples'] <= 0:
        return forecast_profile.copy(), forecast_window_mean
    adjusted_profile = forecast_window_mean * (1.0 + factor_model['hourly_factors'])
    return adjusted_profile.astype(np.float64), forecast_window_mean

bias_adjusted_profiles = {}
bias_rows = []

for unique_id, series_runs in rolling_runs.items():
    ordered_target_dates = sorted(series_runs, key=lambda value: pd.Timestamp(value))

    for target_date in ordered_target_dates:
        run = series_runs[target_date]
        if run.actual_profile is None or len(run.forecast_profile) != SEASON_LENGTH:
            continue

        full_actual = run.actual_profile.copy()
        full_forecast = run.forecast_profile.copy()
        head_actual = full_actual[:BIAS_HEAD_HOURS]
        head_forecast = full_forecast[:BIAS_HEAD_HOURS]
        tail_actual = full_actual[BIAS_HEAD_HOURS:]
        tail_forecast = full_forecast[BIAS_HEAD_HOURS:]

        factor_model = analog_holidays_module.fit_hourly_bias_factor_model(
            neighbor_profiles=run.neighbors2,
            window_hours=SEASON_LENGTH,
            max_analogs=HOURLY_FACTOR_ANALOGS,
        )
        adjusted_full_forecast, forecast_window_mean_38 = _apply_hourly_factor_model(
            full_forecast,
            factor_model,
            expected_hours=SEASON_LENGTH,
        )
        adjusted_head_forecast = adjusted_full_forecast[:BIAS_HEAD_HOURS].copy()
        adjusted_tail_forecast = adjusted_full_forecast[BIAS_HEAD_HOURS:].copy()
        forecast_daily_mean_24 = float(np.mean(tail_forecast))
        predicted_full_bias_profile = adjusted_full_forecast - full_forecast
        predicted_head_bias_mean = float(np.mean(predicted_full_bias_profile[:BIAS_HEAD_HOURS]))
        predicted_tail_bias_mean = float(np.mean(predicted_full_bias_profile[BIAS_HEAD_HOURS:]))
        predicted_full_bias_mean = float(np.mean(predicted_full_bias_profile))

        metrics_14 = _build_window_metrics(head_actual, head_forecast, '14')
        metrics_24 = _build_window_metrics(tail_actual, tail_forecast, '24')
        metrics_38 = _build_window_metrics(full_actual, full_forecast, '38')
        metrics_14_bias_adjusted = _build_window_metrics(head_actual, adjusted_head_forecast, '14_bias_adjusted')
        metrics_24_bias_adjusted = _build_window_metrics(tail_actual, adjusted_tail_forecast, '24_bias_adjusted')
        metrics_38_bias_adjusted = _build_window_metrics(full_actual, adjusted_full_forecast, '38_bias_adjusted')
        predicted_bias_14 = predicted_head_bias_mean
        predicted_bias_24 = predicted_tail_bias_mean
        predicted_bias_38 = predicted_full_bias_mean
        bias_14_prediction_error = predicted_bias_14 - metrics_14['bias_14']
        bias_24_prediction_error = predicted_bias_24 - metrics_24['bias_24']
        bias_38_prediction_error = predicted_bias_38 - metrics_38['bias_38']

        bias_adjusted_profiles[(unique_id, target_date)] = {
            'full_actual': full_actual.copy(),
            'full_forecast': full_forecast.copy(),
            'adjusted_full_forecast': adjusted_full_forecast.copy(),
            'head_actual': head_actual.copy(),
            'head_forecast': head_forecast.copy(),
            'adjusted_head_forecast': adjusted_head_forecast.copy(),
            'tail_actual': tail_actual.copy(),
            'tail_forecast': tail_forecast.copy(),
            'adjusted_tail_forecast': adjusted_tail_forecast.copy(),
            'hourly_adjustment_factors': factor_model['hourly_factors'].copy(),
            'hourly_factor_analog_count': factor_model['train_samples'],
            'hourly_factor_requested_analogs': factor_model['requested_analogs'],
            'hourly_factor_available_neighbors': factor_model['available_neighbor_profiles'],
            'hourly_factor_selected_analogs': factor_model['selected_analogs'],
            'forecast_window_mean_38': forecast_window_mean_38,
            'forecast_daily_mean_24': forecast_daily_mean_24,
        }

        bias_rows.append({
            'unique_id': unique_id,
            'target_date': target_date,
            'bias_head_hours': BIAS_HEAD_HOURS,
            'bias_tail_hours': BIAS_TAIL_HOURS,
            'hourly_factor_analog_count': factor_model['train_samples'],
            'hourly_factor_requested_analogs': factor_model['requested_analogs'],
            'hourly_factor_available_neighbors': factor_model['available_neighbor_profiles'],
            'hourly_factor_selected_analogs': factor_model['selected_analogs'],
            'hourly_factor_mean_abs': factor_model['factor_mean_abs'],
            'forecast_window_mean_38': forecast_window_mean_38,
            'forecast_daily_mean_24': forecast_daily_mean_24,
            **metrics_38,
            **metrics_14,
            **metrics_24,
            **metrics_14_bias_adjusted,
            **metrics_24_bias_adjusted,
            **metrics_38_bias_adjusted,
            'head_bias_mean': metrics_14['bias_14'],
            'tail_bias_mean': metrics_24['bias_24'],
            'bias_model_method': factor_model['method'],
            'bias_train_samples': factor_model['train_samples'],
            'bias_train_window_mean': factor_model['window_mean_train'],
            'bias_train_head_tail_corr': factor_model['head_tail_corr_train'],
            'bias_model_intercept': factor_model['intercept'],
            'bias_model_slope': factor_model['slope'],
            'predicted_head_bias_mean': predicted_head_bias_mean,
            'predicted_tail_bias_mean': predicted_tail_bias_mean,
            'predicted_full_bias_mean': predicted_full_bias_mean,
            'predicted_bias_14': predicted_bias_14,
            'predicted_bias_24': predicted_bias_24,
            'predicted_bias_38': predicted_bias_38,
            'bias_14_prediction_error': bias_14_prediction_error,
            'bias_24_prediction_error': bias_24_prediction_error,
            'bias_38_prediction_error': bias_38_prediction_error,
            'mae_head14': metrics_14['mae_14'],
            'mape_head14_pct': metrics_14['mape_14_pct'],
            'mae_head14_bias_adjusted': metrics_14_bias_adjusted['mae_14_bias_adjusted'],
            'mape_head14_bias_adjusted_pct': metrics_14_bias_adjusted['mape_14_bias_adjusted_pct'],
            'mae_holiday24_raw': metrics_24['mae_24'],
            'mape_holiday24_raw_pct': metrics_24['mape_24_pct'],
            'mae_holiday24_bias_adjusted': metrics_24_bias_adjusted['mae_24_bias_adjusted'],
            'mape_holiday24_bias_adjusted_pct': metrics_24_bias_adjusted['mape_24_bias_adjusted_pct'],
            'mae_window_bias_adjusted': metrics_38_bias_adjusted['mae_38_bias_adjusted'],
            'mape_window_bias_adjusted_pct': metrics_38_bias_adjusted['mape_38_bias_adjusted_pct'],
        })

rolling_bias_adjustment_df = pd.DataFrame(bias_rows)
if rolling_bias_adjustment_df.empty:
    raise ValueError('No complete rolling runs were available to build the hourly-adjustment study.')

rolling_daily_table = rolling_daily_table.merge(
    rolling_bias_adjustment_df,
    on=['unique_id', 'target_date'],
    how='left',
    validate='one_to_one',
)

rolling_daily_table['mae_holiday24_improvement'] = (
    rolling_daily_table['mae_holiday24_raw'] - rolling_daily_table['mae_holiday24_bias_adjusted']
)
rolling_daily_table['mape_holiday24_improvement_pct'] = (
    rolling_daily_table['mape_holiday24_raw_pct'] - rolling_daily_table['mape_holiday24_bias_adjusted_pct']
)
rolling_daily_table['mape_head14_improvement_pct'] = (
    rolling_daily_table['mape_head14_pct'] - rolling_daily_table['mape_head14_bias_adjusted_pct']
)
rolling_daily_table['mape_window_improvement_pct'] = (
    rolling_daily_table['mape_window_pct'] - rolling_daily_table['mape_window_bias_adjusted_pct']
)
rolling_daily_table['bias_14_prediction_abs_error'] = rolling_daily_table['bias_14_prediction_error'].abs()
rolling_daily_table['bias_24_prediction_abs_error'] = rolling_daily_table['bias_24_prediction_error'].abs()
rolling_daily_table['bias_38_prediction_abs_error'] = rolling_daily_table['bias_38_prediction_error'].abs()

if 'batch_result_2025_all' in globals() and batch_result_2025_all:
    for series_unique_id, series_batch_result in batch_result_2025_all.items():
        series_batch_result.results_df = (
            rolling_daily_table
            .loc[rolling_daily_table['unique_id'] == series_unique_id]
            .reset_index(drop=True)
        )
        metric_summary_columns = [
            column for column in [
                'mae_38',
                'mape_38_pct',
                'mpe_38_pct',
                'bias_38',
                'mae_14',
                'mape_14_pct',
                'mpe_14_pct',
                'bias_14',
                'mae_24',
                'mape_24_pct',
                'mpe_24_pct',
                'bias_24',
                'predicted_bias_38',
                'bias_38_prediction_error',
                'predicted_bias_24',
                'bias_24_prediction_error',
                'hourly_factor_analog_count',
                'hourly_factor_requested_analogs',
                'hourly_factor_available_neighbors',
                'hourly_factor_selected_analogs',
                'hourly_factor_mean_abs',
                'forecast_window_mean_38',
                'forecast_daily_mean_24',
                'mae_holiday24_bias_adjusted',
                'mape_holiday24_bias_adjusted_pct',
                'mpe_24_bias_adjusted_pct',
                'bias_24_bias_adjusted',
                'mae_holiday24_raw',
                'mape_holiday24_raw_pct',
                'mae_window_bias_adjusted',
                'mape_window_bias_adjusted_pct',
                'mpe_38_bias_adjusted_pct',
                'bias_38_bias_adjusted',
            ]
            if column in series_batch_result.results_df.columns
        ]
        series_batch_result.metric_summary_df = series_batch_result.results_df[metric_summary_columns].describe(include='all')
bias_summary_df = (
    rolling_daily_table
    .groupby('unique_id', dropna=False)
    .agg(
        rows=('target_date', 'size'),
        rows_with_train_history=('bias_train_samples', lambda values: int((values > 0).sum())),
        median_bias_train_samples=('bias_train_samples', 'median'),
        mean_mae_38=('mae_38', 'mean'),
        mean_mae_14=('mae_14', 'mean'),
        mean_mae_24=('mae_24', 'mean'),
        mean_mape_38_pct=('mape_38_pct', 'mean'),
        mean_mape_14_pct=('mape_14_pct', 'mean'),
        mean_mape_24_pct=('mape_24_pct', 'mean'),
        mean_mpe_38_pct=('mpe_38_pct', 'mean'),
        mean_mpe_14_pct=('mpe_14_pct', 'mean'),
        mean_mpe_24_pct=('mpe_24_pct', 'mean'),
        mean_bias_38=('bias_38', 'mean'),
        mean_bias_14=('bias_14', 'mean'),
        mean_bias_24=('bias_24', 'mean'),
        mean_hourly_factor_analog_count=('hourly_factor_analog_count', 'mean'),
        mean_hourly_factor_requested_analogs=('hourly_factor_requested_analogs', 'mean'),
        mean_hourly_factor_available_neighbors=('hourly_factor_available_neighbors', 'mean'),
        mean_hourly_factor_selected_analogs=('hourly_factor_selected_analogs', 'mean'),
        mean_hourly_factor_mean_abs=('hourly_factor_mean_abs', 'mean'),
        mean_predicted_bias_38=('predicted_bias_38', 'mean'),
        mean_bias_38_prediction_error=('bias_38_prediction_error', 'mean'),
        mean_predicted_bias_24=('predicted_bias_24', 'mean'),
        mean_bias_24_prediction_error=('bias_24_prediction_error', 'mean'),
    )
    .reset_index()
)

eligible_bias_rows = rolling_daily_table['bias_train_samples'] > 0
bias_linkage_summary_rows = []
for series_unique_id, series_df in rolling_daily_table.groupby('unique_id', dropna=False):
    eligible_series_df = series_df.loc[series_df['bias_train_samples'] > 0].copy()
    bias_linkage_summary_rows.append({
        'unique_id': series_unique_id,
        'rows_with_train_history': int(len(eligible_series_df)),
        'corr_bias_14_vs_24': _safe_corr(eligible_series_df['bias_14'], eligible_series_df['bias_24']),
        'corr_mpe_14_vs_24_pct': _safe_corr(eligible_series_df['mpe_14_pct'], eligible_series_df['mpe_24_pct']),
        'mean_hourly_factor_analog_count': float(eligible_series_df['hourly_factor_analog_count'].mean()) if not eligible_series_df.empty else np.nan,
        'mean_hourly_factor_requested_analogs': float(eligible_series_df['hourly_factor_requested_analogs'].mean()) if not eligible_series_df.empty else np.nan,
        'mean_hourly_factor_available_neighbors': float(eligible_series_df['hourly_factor_available_neighbors'].mean()) if not eligible_series_df.empty else np.nan,
        'mean_hourly_factor_selected_analogs': float(eligible_series_df['hourly_factor_selected_analogs'].mean()) if not eligible_series_df.empty else np.nan,
        'mean_hourly_factor_mean_abs': float(eligible_series_df['hourly_factor_mean_abs'].mean()) if not eligible_series_df.empty else np.nan,
        'mean_predicted_bias_38': float(eligible_series_df['predicted_bias_38'].mean()) if not eligible_series_df.empty else np.nan,
        'mean_actual_bias_38': float(eligible_series_df['bias_38'].mean()) if not eligible_series_df.empty else np.nan,
        'mae_bias_prediction_38': float(np.mean(np.abs(eligible_series_df['bias_38_prediction_error']))) if not eligible_series_df.empty else np.nan,
        'mean_bias_prediction_error_38': float(eligible_series_df['bias_38_prediction_error'].mean()) if not eligible_series_df.empty else np.nan,
    })
bias_linkage_summary_df = pd.DataFrame(bias_linkage_summary_rows)

overall_bias_delta_df = pd.DataFrame([{
    'rows_total': int(len(rolling_daily_table)),
    'rows_with_train_history': int(eligible_bias_rows.sum()),
    'mean_mape_38_pct': float(rolling_daily_table.loc[eligible_bias_rows, 'mape_38_pct'].mean()),
    'mean_mape_14_pct': float(rolling_daily_table.loc[eligible_bias_rows, 'mape_14_pct'].mean()),
    'mean_mape_24_pct': float(rolling_daily_table.loc[eligible_bias_rows, 'mape_24_pct'].mean()),
    'mean_mae_38': float(rolling_daily_table.loc[eligible_bias_rows, 'mae_38'].mean()),
    'mean_mae_14': float(rolling_daily_table.loc[eligible_bias_rows, 'mae_14'].mean()),
    'mean_mae_24': float(rolling_daily_table.loc[eligible_bias_rows, 'mae_24'].mean()),
    'mean_mpe_38_pct': float(rolling_daily_table.loc[eligible_bias_rows, 'mpe_38_pct'].mean()),
    'mean_mpe_14_pct': float(rolling_daily_table.loc[eligible_bias_rows, 'mpe_14_pct'].mean()),
    'mean_mpe_24_pct': float(rolling_daily_table.loc[eligible_bias_rows, 'mpe_24_pct'].mean()),
    'mean_bias_38': float(rolling_daily_table.loc[eligible_bias_rows, 'bias_38'].mean()),
    'mean_bias_14': float(rolling_daily_table.loc[eligible_bias_rows, 'bias_14'].mean()),
    'mean_bias_24': float(rolling_daily_table.loc[eligible_bias_rows, 'bias_24'].mean()),
    'mean_hourly_factor_analog_count': float(rolling_daily_table.loc[eligible_bias_rows, 'hourly_factor_analog_count'].mean()),
    'mean_hourly_factor_requested_analogs': float(rolling_daily_table.loc[eligible_bias_rows, 'hourly_factor_requested_analogs'].mean()),
    'mean_hourly_factor_available_neighbors': float(rolling_daily_table.loc[eligible_bias_rows, 'hourly_factor_available_neighbors'].mean()),
    'mean_hourly_factor_selected_analogs': float(rolling_daily_table.loc[eligible_bias_rows, 'hourly_factor_selected_analogs'].mean()),
    'mean_hourly_factor_mean_abs': float(rolling_daily_table.loc[eligible_bias_rows, 'hourly_factor_mean_abs'].mean()),
    'mean_predicted_bias_38': float(rolling_daily_table.loc[eligible_bias_rows, 'predicted_bias_38'].mean()),
    'corr_bias_14_vs_24': _safe_corr(rolling_daily_table.loc[eligible_bias_rows, 'bias_14'], rolling_daily_table.loc[eligible_bias_rows, 'bias_24']),
    'holiday24_improved_rows': int((rolling_daily_table['mape_holiday24_improvement_pct'] > 0).sum()),
    'window38_improved_rows': int((rolling_daily_table['mape_window_improvement_pct'] > 0).sum()),
    'window38_worsened_rows': int((rolling_daily_table['mape_window_improvement_pct'] < 0).sum()),
}])

display(overall_bias_delta_df)
display(bias_summary_df)
display(bias_linkage_summary_df)
rolling_daily_table[[
    'unique_id',
    'target_date',
    'analog_cluster',
    'bias_train_samples',
    'bias_model_method',
    'hourly_factor_analog_count',
    'hourly_factor_requested_analogs',
    'hourly_factor_available_neighbors',
    'hourly_factor_selected_analogs',
    'hourly_factor_mean_abs',
    'forecast_window_mean_38',
    'forecast_daily_mean_24',
    'mae_38',
    'mape_38_pct',
    'mpe_38_pct',
    'bias_38',
    'mae_14',
    'mape_14_pct',
    'mpe_14_pct',
    'bias_14',
    'mae_24',
    'mape_24_pct',
    'mpe_24_pct',
    'bias_24',
    'predicted_bias_38',
    'bias_38_prediction_error',
    'predicted_bias_24',
    'bias_24_prediction_error',
    'mae_24_bias_adjusted',
    'mape_24_bias_adjusted_pct',
    'mpe_24_bias_adjusted_pct',
    'bias_24_bias_adjusted',
    'mae_38_bias_adjusted',
    'mape_38_bias_adjusted_pct',
    'mpe_38_bias_adjusted_pct',
    'bias_38_bias_adjusted',
    'mape_holiday24_improvement_pct',
    'mape_window_bias_adjusted_pct',
    'mape_window_improvement_pct',
]].sort_values(['unique_id', 'target_date']).reset_index(drop=True)

,rows_total,rows_with_train_history,mean_mape_38_pct,mean_mape_14_pct,mean_mape_24_pct,mean_mae_38,mean_mae_14,mean_mae_24,mean_mpe_38_pct,mean_mpe_14_pct,...,mean_hourly_factor_analog_count,mean_hourly_factor_requested_analogs,mean_hourly_factor_available_neighbors,mean_hourly_factor_selected_analogs,mean_hourly_factor_mean_abs,mean_predicted_bias_38,corr_bias_14_vs_24,holiday24_improved_rows,window38_improved_rows,window38_worsened_rows
0,171,171,7.407596,6.108897,8.165171,789.185321,683.337633,850.929805,-0.450892,-0.153033,...,3.842105,4.0,4.929825,3.842105,0.07365,5.106460e-13,0.699365,64,60,111


,unique_id,rows,rows_with_train_history,median_bias_train_samples,mean_mae_38,mean_mae_14,mean_mae_24,mean_mape_38_pct,mean_mape_14_pct,mean_mape_24_pct,...,mean_bias_24,mean_hourly_factor_analog_count,mean_hourly_factor_requested_analogs,mean_hourly_factor_available_neighbors,mean_hourly_factor_selected_analogs,mean_hourly_factor_mean_abs,mean_predicted_bias_38,mean_bias_38_prediction_error,mean_predicted_bias_24,mean_bias_24_prediction_error
0,ERCOT_demand_COAST,19,19,4.0,959.398129,876.565322,1007.717266,6.779019,6.004597,7.230765,...,51.238747,4.000000,4.0,4.105263,4.000000,0.074388,-3.023251e-13,10.223559,-53.791837,-105.030584
1,ERCOT_demand_EAST,19,19,4.0,199.310726,153.658372,225.941265,10.668213,7.988466,12.231399,...,33.217263,4.000000,4.0,5.473684,4.000000,0.082757,-6.298440e-16,-31.602261,-5.979530,-39.196793
2,ERCOT_demand_ERCOT,19,19,4.0,2984.186911,2540.568159,3242.964516,5.437644,4.500367,5.984388,...,212.558082,4.000000,4.0,4.894737,4.000000,0.075948,4.111621e-12,-184.502011,-362.610205,-575.168287
3,ERCOT_demand_FWEST,19,19,4.0,212.275681,210.311476,213.421468,2.794534,2.763646,2.812551,...,-2.823372,4.000000,4.0,6.578947,4.000000,0.022071,3.653095e-14,-36.299681,-27.543671,-24.720299
4,ERCOT_demand_NCENT,19,19,2.0,1448.181591,1362.323259,1498.265619,9.998705,8.676503,10.769990,...,-243.916678,2.842105,4.0,3.210526,2.842105,0.087343,3.501933e-13,175.921090,-118.940858,124.975821
5,ERCOT_demand_NORTH,19,19,4.0,129.225820,115.713010,137.108292,7.844080,6.991059,8.341675,...,14.315675,4.000000,4.0,5.789474,4.000000,0.069077,-7.778573e-14,-8.905585,-11.376324,-25.691998
6,ERCOT_demand_SCENT,19,19,4.0,716.690271,514.873737,834.416582,8.541696,6.226938,9.891971,...,-108.347409,3.947368,4.0,4.631579,3.947368,0.088814,2.380810e-13,97.610912,-66.597824,41.749585
7,ERCOT_demand_SOUTH,19,19,4.0,369.329510,313.255369,402.039426,8.541907,7.395865,9.210432,...,35.236293,3.947368,4.0,5.052632,3.947368,0.094697,1.965113e-13,1.262293,-20.811137,-56.047430
8,ERCOT_demand_WEST,19,19,4.0,84.069247,62.769989,96.493813,6.062572,4.432635,7.013368,...,-4.749476,3.842105,4.0,4.631579,3.842105,0.067758,4.361670e-14,-0.222602,-2.648147,2.101329


,unique_id,rows_with_train_history,corr_bias_14_vs_24,corr_mpe_14_vs_24_pct,mean_hourly_factor_analog_count,mean_hourly_factor_requested_analogs,mean_hourly_factor_available_neighbors,mean_hourly_factor_selected_analogs,mean_hourly_factor_mean_abs,mean_predicted_bias_38,mean_actual_bias_38,mae_bias_prediction_38,mean_bias_prediction_error_38
0,ERCOT_demand_COAST,19,0.789995,0.815471,4.000000,4.0,4.105263,4.000000,0.074388,-3.023251e-13,-10.223559,793.593668,10.223559
1,ERCOT_demand_EAST,19,0.842870,0.798987,4.000000,4.0,5.473684,4.000000,0.082757,-6.298440e-16,31.602261,167.448689,-31.602261
2,ERCOT_demand_ERCOT,19,0.740805,0.784457,4.000000,4.0,4.894737,4.000000,0.075948,4.111621e-12,184.502011,2336.955272,-184.502011
3,ERCOT_demand_FWEST,19,0.362885,0.354856,4.000000,4.0,6.578947,4.000000,0.022071,3.653095e-14,36.299681,102.939791,-36.299681
4,ERCOT_demand_NCENT,19,0.483520,0.556223,2.842105,4.0,3.210526,2.842105,0.087343,3.501933e-13,-175.921090,1113.030381,175.921090
5,ERCOT_demand_NORTH,19,0.677158,0.687534,4.000000,4.0,5.789474,4.000000,0.069077,-7.778573e-14,8.905585,105.332449,-8.905585
6,ERCOT_demand_SCENT,19,0.797802,0.839433,3.947368,4.0,4.631579,3.947368,0.088814,2.380810e-13,-97.610912,574.787682,97.610912
7,ERCOT_demand_SOUTH,19,0.822784,0.851389,3.947368,4.0,5.052632,3.947368,0.094697,1.965113e-13,-1.262293,325.063252,1.262293
8,ERCOT_demand_WEST,19,0.732556,0.717295,3.842105,4.0,4.631579,3.842105,0.067758,4.361670e-14,0.222602,69.914907,-0.222602


,unique_id,target_date,analog_cluster,bias_train_samples,bias_model_method,hourly_factor_analog_count,hourly_factor_requested_analogs,hourly_factor_available_neighbors,hourly_factor_selected_analogs,hourly_factor_mean_abs,...,mape_24_bias_adjusted_pct,mpe_24_bias_adjusted_pct,bias_24_bias_adjusted,mae_38_bias_adjusted,mape_38_bias_adjusted_pct,mpe_38_bias_adjusted_pct,bias_38_bias_adjusted,mape_holiday24_improvement_pct,mape_window_bias_adjusted_pct,mape_window_improvement_pct
0,ERCOT_demand_COAST,2025-01-01,O,4,hourly_window_mean_factor_top_available_analogs,4,4,4,4,0.053194,...,9.437002,-9.437002,-1052.475377,914.398791,8.080082,-8.080082,-914.398791,1.154065,8.080082,0.082621
1,ERCOT_demand_COAST,2025-01-20,M,4,hourly_window_mean_factor_top_available_analogs,4,4,5,4,0.046418,...,26.007269,26.007269,4349.416913,3761.647006,23.294973,23.294973,3761.647006,0.647912,23.294973,-0.157027
2,ERCOT_demand_COAST,2025-02-17,P,4,hourly_window_mean_factor_top_available_analogs,4,4,4,4,0.066026,...,7.341895,3.149836,409.346191,794.991685,6.260492,1.564285,210.783194,-1.634192,6.260492,-1.331407
3,ERCOT_demand_COAST,2025-03-02,R,4,hourly_window_mean_factor_top_available_analogs,4,4,4,4,0.075968,...,4.877865,2.492098,269.806196,494.370503,4.350563,2.059243,235.320999,-1.194889,4.350563,-0.482566
4,ERCOT_demand_COAST,2025-04-21,Q,4,hourly_window_mean_factor_top_available_analogs,4,4,4,4,0.086528,...,6.587861,6.587861,943.676310,794.722595,5.412243,5.245215,771.202469,-1.155061,5.412243,-0.300343
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,ERCOT_demand_WEST,2025-12-25,F,4,hourly_window_mean_factor_top_available_analogs,4,4,4,4,0.033879,...,5.760487,-0.441808,-2.030097,71.542750,5.506458,1.397730,21.436666,-2.790729,5.506458,-2.199882
167,ERCOT_demand_WEST,2026-01-01,O,4,hourly_window_mean_factor_top_available_analogs,4,4,4,4,0.045344,...,7.935691,-7.935691,-105.514999,75.745559,5.671023,-5.299218,-70.911708,-0.181442,5.671023,-0.250695
168,ERCOT_demand_WEST,2026-01-19,M,4,hourly_window_mean_factor_top_available_analogs,4,4,4,4,0.067608,...,10.557528,-10.557528,-170.140964,152.503018,9.558534,-9.558534,-152.503018,-0.568161,9.558534,0.259994
169,ERCOT_demand_WEST,2026-02-16,P,4,hourly_window_mean_factor_top_available_analogs,4,4,4,4,0.044759,...,5.843020,5.843020,79.648330,82.333624,6.047253,6.047253,82.333624,1.207854,6.047253,0.165653


In [9]:
if 'rolling_daily_table' not in globals() or rolling_daily_table.empty:
    raise ValueError('Run Cell 10 first to build the bias-adjusted rolling_daily_table before the segmented error analysis.')
if 'mape_holiday24_bias_adjusted_pct' not in rolling_daily_table.columns:
    raise ValueError('Run Cell 10 first so the adjusted holiday24 KPI is available for the segmented error analysis.')

if 'selector_features_df' not in globals():
    selector_features_df = pd.read_csv(SELECTOR_FEATURES_PATH, parse_dates=['date'])
    selector_features_df['date'] = pd.to_datetime(selector_features_df['date']).dt.normalize()
if 'unique_id' not in selector_features_df.columns:
    raise ValueError(
        f'{SELECTOR_FEATURES_PATH.name} must contain a unique_id column. '
        'Re-export the selector from M_identify_holidays.ipynb.'
    )

selector_analysis_df = selector_features_df.copy().rename(
    columns={
        'analog_cluster': 'selector_analog_cluster',
        'day_class_code': 'selector_day_class_code',
        'daily_profile_cluster_id': 'daily_profile_cluster_id_raw',
        'event_profile_cluster_id': 'event_profile_cluster_id_raw',
    }
)
selector_analysis_df['unique_id'] = selector_analysis_df['unique_id'].astype(str)
selector_analysis_df['target_date'] = pd.to_datetime(selector_analysis_df['date']).dt.date.astype(str)
selector_analysis_df = selector_analysis_df.drop(columns=['date']).drop_duplicates(
    subset=['unique_id', 'target_date'],
    keep='last',
)

error_analysis_df = rolling_daily_table.copy()
error_analysis_df['unique_id'] = error_analysis_df['unique_id'].astype(str)
error_analysis_df['target_date'] = pd.to_datetime(error_analysis_df['target_date']).dt.date.astype(str)
error_analysis_df = error_analysis_df.merge(
    selector_analysis_df,
    on=['unique_id', 'target_date'],
    how='left',
    validate='many_to_one',
)

def _resolve_weekend_like_category(row):
    best_matching_weekday = row.get('best_matching_weekday')
    daily_profile_archetype = row.get('daily_profile_archetype')
    best_text = '' if pd.isna(best_matching_weekday) else str(best_matching_weekday).strip().lower()
    archetype_text = '' if pd.isna(daily_profile_archetype) else str(daily_profile_archetype).strip().lower()
    if best_text == 'saturday':
        return 'Saturday'
    if best_text == 'sunday':
        return 'Sunday'
    if 'saturday' in archetype_text:
        return 'Saturday-like'
    if 'sunday' in archetype_text:
        return 'Sunday-like'
    return 'Other / unclear'

error_analysis_df['weekend_like_category'] = error_analysis_df.apply(_resolve_weekend_like_category, axis=1)
error_analysis_df['k_label'] = error_analysis_df['k'].round().astype('Int64').astype(str)
error_analysis_df['scale_method_label'] = error_analysis_df['scale_method'].astype('string').fillna('None')
error_analysis_df['selected_analogs_label'] = error_analysis_df['selected_analogs'].round().astype('Int64').astype(str)
error_analysis_df['eligible_tuning_dates_label'] = error_analysis_df['eligible_tuning_dates'].round().astype('Int64').astype(str)
error_analysis_df['target_has_complete_profile_label'] = error_analysis_df['target_has_complete_profile'].map({True: 'complete', False: 'incomplete'})
error_analysis_df['fail_label'] = error_analysis_df['fail'].map({True: 'fail', False: 'ok'})
error_analysis_df['selector_day_class_code_label'] = error_analysis_df['selector_day_class_code'].astype('string')
error_analysis_df['daily_profile_cluster_id_label'] = error_analysis_df['daily_profile_cluster_id_raw'].astype('Int64').astype(str)
error_analysis_df['event_profile_cluster_id_label'] = error_analysis_df['event_profile_cluster_id_raw'].astype('Int64').astype(str)
error_analysis_df['is_fixed_date_label'] = error_analysis_df['is_fixed_date'].map({True: 'fixed', False: 'not_fixed'})
error_analysis_df['is_observed_monday_rule_label'] = error_analysis_df['is_observed_monday_rule'].map({True: 'observed_monday', False: 'not_observed_monday'})
PRIMARY_KPI = 'mape_holiday24_bias_adjusted_pct'
PRIMARY_KPI_LABEL = 'Holiday 24h bias-adjusted MAPE %'

model_segment_columns = [
    'unique_id',
    'typereg',
    'typedist',
    'k_label',
    'scale_method_label',
    'selected_analogs_label',
    'eligible_tuning_dates_label',
    'target_has_complete_profile_label',
    'fail_label',
]
selector_segment_columns = [
    'selector_analog_cluster',
    'holiday_name',
    'anchor_holiday_name',
    'holiday_day_type',
    'weekday_name',
    'selector_day_class_code_label',
    'day_class_name',
    'season',
    'date_rule',
    'is_fixed_date_label',
    'is_observed_monday_rule_label',
    'best_matching_weekday',
    'weekend_like_category',
    'daily_profile_cluster',
    'daily_profile_cluster_id_label',
    'daily_profile_archetype',
    'event_profile_cluster',
    'event_profile_cluster_id_label',
]
segment_columns = [
    column
    for column in model_segment_columns + selector_segment_columns
    if column in error_analysis_df.columns
]

def build_segment_metric_stats(df, feature, metric, min_count=2):
    working_df = df[[feature, metric]].copy().dropna()
    if working_df.empty:
        return pd.DataFrame()
    working_df[feature] = working_df[feature].astype(str)
    stats_df = (
        working_df.groupby(feature, dropna=False)[metric]
        .agg(
            count='size',
            mean='mean',
            median='median',
            q25=lambda values: values.quantile(0.25),
            q75=lambda values: values.quantile(0.75),
            max='max',
        )
        .reset_index()
    )
    stats_df = stats_df.loc[stats_df['count'] >= min_count].copy()
    if stats_df.empty:
        return stats_df
    return stats_df.sort_values(['median', 'mean', 'count'], ascending=[False, False, False]).reset_index(drop=True)

def build_feature_hotspots(df, features, metric='mape_holiday24_bias_adjusted_pct', min_count=2):
    hotspot_rows = []
    for feature in features:
        feature_stats_df = build_segment_metric_stats(df, feature, metric=metric, min_count=min_count)
        if feature_stats_df.empty:
            continue
        worst_row = feature_stats_df.iloc[0]
        hotspot_rows.append({
            'feature': feature,
            'worst_segment': worst_row[feature],
            'count': int(worst_row['count']),
            'mean': float(worst_row['mean']),
            'median': float(worst_row['median']),
            'q75': float(worst_row['q75']),
            'max': float(worst_row['max']),
        })
    if not hotspot_rows:
        return pd.DataFrame()
    return pd.DataFrame(hotspot_rows).sort_values(['median', 'q75', 'count'], ascending=[False, False, False]).reset_index(drop=True)

def build_all_segment_stats(df, features, metric='mape_holiday24_bias_adjusted_pct', min_count=2):
    stats_frames = []
    for feature in features:
        feature_stats_df = build_segment_metric_stats(df, feature, metric=metric, min_count=min_count)
        if feature_stats_df.empty:
            continue
        feature_stats_df = feature_stats_df.rename(columns={feature: 'segment_value'})
        feature_stats_df.insert(0, 'feature', feature)
        stats_frames.append(feature_stats_df)
    if not stats_frames:
        return pd.DataFrame()
    return pd.concat(stats_frames, ignore_index=True).sort_values(
        ['median', 'q75', 'count'],
        ascending=[False, False, False],
    ).reset_index(drop=True)

def _ordered_levels(stats_df, feature, max_levels):
    if feature in {
        'k_label',
        'selected_analogs_label',
        'eligible_tuning_dates_label',
        'selector_day_class_code_label',
        'daily_profile_cluster_id_label',
        'event_profile_cluster_id_label',
    }:
        filtered = stats_df.loc[stats_df[feature] != '<NA>', feature].tolist()
        filtered = sorted(filtered, key=lambda value: float(value))
        return filtered[:max_levels]
    return stats_df.head(max_levels)[feature].tolist()

def plot_segmented_error_boxplots(df, features, metric='mape_holiday24_bias_adjusted_pct', min_count=2, max_levels=10, ncols=2):
    valid_features = [feature for feature in features if feature in df.columns]
    if not valid_features:
        raise ValueError('No valid segment columns were found for the boxplot analysis.')

    nrows = int(np.ceil(len(valid_features) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4.8 * nrows), constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()

    for axis, feature in zip(axes, valid_features):
        stats_df = build_segment_metric_stats(df, feature, metric=metric, min_count=min_count)
        if stats_df.empty:
            axis.set_visible(False)
            continue

        levels = _ordered_levels(stats_df, feature, max_levels=max_levels)
        plot_df = df.loc[df[feature].astype(str).isin(levels), [feature, metric]].copy().dropna()
        if plot_df.empty:
            axis.set_visible(False)
            continue

        plot_df[feature] = pd.Categorical(plot_df[feature].astype(str), categories=levels, ordered=True)
        plot_df.boxplot(column=metric, by=feature, ax=axis, rot=35, grid=False)
        counts = stats_df.set_index(feature).loc[levels, 'count'].astype(int).tolist()
        axis.set_title(feature)
        axis.set_xlabel('')
        axis.set_ylabel(metric)
        axis.set_xticklabels([f'{level}\n(n={count})' for level, count in zip(levels, counts)], rotation=35, ha='right')

    for axis in axes[len(valid_features):]:
        axis.set_visible(False)

    fig.suptitle(f'Error boxplots by segment | metric={metric}', y=1.01, fontsize=14)
    return fig, axes

analysis_min_count = 2
boxplot_max_levels = 10

mape_feature_hotspots_df = build_feature_hotspots(
    error_analysis_df,
    segment_columns,
    metric=PRIMARY_KPI,
    min_count=analysis_min_count,
    )
mae_feature_hotspots_df = build_feature_hotspots(
    error_analysis_df,
    segment_columns,
    metric='mae_holiday24_bias_adjusted',
    min_count=analysis_min_count,
    )
segment_mape_stats_df = build_all_segment_stats(
    error_analysis_df,
    segment_columns,
    metric=PRIMARY_KPI,
    min_count=analysis_min_count,
    )

print(
    f'Rows analyzed: {len(error_analysis_df)} | series: {error_analysis_df["unique_id"].nunique()} | '
    f'target dates: {error_analysis_df["target_date"].nunique()}'
)
display(
    error_analysis_df[[
        'unique_id',
        'target_date',
        'holiday_label',
        'selector_analog_cluster',
        'holiday_day_type',
        'typereg',
        'typedist',
        'k',
        'scale_method_label',
        'weekend_like_category',
        'mae_holiday24_bias_adjusted',
        'mape_holiday24_raw_pct',
        'mape_holiday24_bias_adjusted_pct',
        'mape_holiday24_improvement_pct',
    ]].head(12)
)

print(f'Worst segment by feature | {PRIMARY_KPI_LABEL}')
display(mape_feature_hotspots_df.head(20))

print('Worst segment by feature | MAE')
display(mae_feature_hotspots_df.head(20))

print(f'All segment stats ranked by median {PRIMARY_KPI_LABEL}')
display(segment_mape_stats_df.head(50))

fig_model_segments, _ = plot_segmented_error_boxplots(
    error_analysis_df,
    model_segment_columns,
    metric=PRIMARY_KPI,
    min_count=analysis_min_count,
    max_levels=boxplot_max_levels,
    )
export_figure_pdf(fig_model_segments, 'error_boxplots_by_model_segment')
plt.show()

fig_selector_segments, _ = plot_segmented_error_boxplots(
    error_analysis_df,
    selector_segment_columns,
    metric=PRIMARY_KPI,
    min_count=analysis_min_count,
    max_levels=boxplot_max_levels,
    )
export_figure_pdf(fig_selector_segments, 'error_boxplots_by_selector_segment')
plt.show()

Rows analyzed: 171 | series: 9 | target dates: 19


,unique_id,target_date,holiday_label,selector_analog_cluster,holiday_day_type,typereg,typedist,k,scale_method_label,weekend_like_category,mae_holiday24_bias_adjusted,mape_holiday24_raw_pct,mape_holiday24_bias_adjusted_pct,mape_holiday24_improvement_pct
0,ERCOT_demand_COAST,2025-01-01,New Year's Day,O,H2,RidgeReg,pearson,4,minmax,Sunday,1052.475377,10.591067,9.437002,1.154065
1,ERCOT_demand_COAST,2025-01-20,Martin Luther King Jr. Day,M,H2,RidgeReg,pearson,5,minmax,Other / unclear,4349.416913,26.655181,26.007269,0.647912
2,ERCOT_demand_COAST,2025-02-17,Presidents' Day,P,H2,PLS,pearson,4,minmax,Other / unclear,946.240320,5.707702,7.341895,-1.634192
3,ERCOT_demand_COAST,2025-03-02,Texas Independence Day,R,H2,PLS,pearson,4,minmax,Other / unclear,532.131404,3.682976,4.877865,-1.194889
4,ERCOT_demand_COAST,2025-04-21,San Jacinto Day,Q,H2,PLS,pearson,4,minmax,Other / unclear,943.676310,5.432800,6.587861,-1.155061
5,ERCOT_demand_COAST,2025-05-26,Memorial Day,N,H2,PLS,pearson,4,minmax,Saturday,1810.040280,9.552354,11.515896,-1.963542
6,ERCOT_demand_COAST,2025-06-19,Juneteenth National Independence Day,J,H2,PLS,pearson,4,minmax,Other / unclear,678.204878,2.600107,4.042784,-1.442677
7,ERCOT_demand_COAST,2025-07-04,Independence Day,I,H2,PLS,pearson,4,minmax,Other / unclear,787.285990,6.126108,5.000505,1.125603
8,ERCOT_demand_COAST,2025-08-27,Lyndon B. Johnson Day,L,H2,PLS,pearson,4,minmax,Other / unclear,564.216533,3.145667,3.251150,-0.105483
9,ERCOT_demand_COAST,2025-09-01,Labor Day,K,H2,PLS,pearson,4,minmax,Other / unclear,625.474200,4.804930,3.939865,0.865065


Worst segment by feature | Holiday 24h bias-adjusted MAPE %


,feature,worst_segment,count,mean,median,q75,max
0,selector_analog_cluster,O,18,12.857633,12.061290,16.448787,25.962644
1,holiday_name,New Year's Day,18,12.857633,12.061290,16.448787,25.962644
2,anchor_holiday_name,New Year's Day,18,12.857633,12.061290,16.448787,25.962644
3,k_label,3,5,11.839060,11.470369,13.937750,16.661607
4,selected_analogs_label,3,5,11.839060,11.470369,13.937750,16.661607
5,daily_profile_archetype,Saturday-like,7,11.369375,10.831398,13.185052,27.769149
6,unique_id,ERCOT_demand_EAST,19,11.670747,10.265912,18.655480,23.739923
7,best_matching_weekday,Sunday,25,10.583788,9.785569,14.031466,27.769149
8,weekend_like_category,Sunday,25,10.583788,9.785569,14.031466,27.769149
9,daily_profile_cluster,A,31,10.394073,9.278010,13.984608,27.769149


Worst segment by feature | MAE


...

,feature,segment_value,count,mean,median,q25,q75,max
0,selector_analog_cluster,O,18,12.857633,12.061290,8.292347,16.448787,25.962644
1,holiday_name,New Year's Day,18,12.857633,12.061290,8.292347,16.448787,25.962644
2,anchor_holiday_name,New Year's Day,18,12.857633,12.061290,8.292347,16.448787,25.962644
3,selector_analog_cluster,N,9,14.499069,11.515896,10.307256,21.870951,27.769149
4,holiday_name,Memorial Day,9,14.499069,11.515896,10.307256,21.870951,27.769149
5,anchor_holiday_name,Memorial Day,9,14.499069,11.515896,10.307256,21.870951,27.769149
6,k_label,3,5,11.839060,11.470369,9.362314,13.937750,16.661607
7,selected_analogs_label,3,5,11.839060,11.470369,9.362314,13.937750,16.661607
8,daily_profile_archetype,Saturday-like,7,11.369375,10.831398,6.066140,13.185052,27.769149
9,selector_analog_cluster,M,18,13.135484,10.642267,6.456255,19.007589,37.141159


Saved PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/error_boxplots_by_model_segment.pdf
Updated multipage PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/all_holiday_graphs_ercot.pdf


/tmp/ipykernel_2652174/2652452214.py:276: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Saved PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/error_boxplots_by_selector_segment.pdf
Updated multipage PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/all_holiday_graphs_ercot.pdf


/tmp/ipykernel_2652174/2652452214.py:286: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [10]:
if 'rolling_runs' not in globals() or not rolling_runs:
    raise ValueError('Run Cell 9 first to build rolling_runs before the OOS action-priority study.')

if 'rolling_daily_table' not in globals() or rolling_daily_table.empty:
    raise ValueError('Run Cell 9 first to build rolling_daily_table before the OOS action-priority study.')

if 'selector_features_df' not in globals():
    selector_features_df = pd.read_csv(SELECTOR_FEATURES_PATH, parse_dates=['date'])
    selector_features_df['date'] = pd.to_datetime(selector_features_df['date']).dt.normalize()
if 'unique_id' not in selector_features_df.columns:
    raise ValueError(
        f'{SELECTOR_FEATURES_PATH.name} must contain a unique_id column. '
        'Re-export the selector from M_identify_holidays.ipynb.'
    )

selector_priority_df = selector_features_df.copy().rename(
    columns={
        'analog_cluster': 'selector_analog_cluster',
        'day_class_code': 'selector_day_class_code',
        'daily_profile_cluster_id': 'daily_profile_cluster_id_raw',
        'event_profile_cluster_id': 'event_profile_cluster_id_raw',
    }
)
selector_priority_df['unique_id'] = selector_priority_df['unique_id'].astype(str)
selector_priority_df['target_date'] = pd.to_datetime(selector_priority_df['date']).dt.date.astype(str)
selector_priority_df = selector_priority_df.drop(columns=['date']).drop_duplicates(
    subset=['unique_id', 'target_date'],
    keep='last',
)

def _resolve_weekend_like_category_priority(row):
    best_matching_weekday = row.get('best_matching_weekday')
    daily_profile_archetype = row.get('daily_profile_archetype')
    best_text = '' if pd.isna(best_matching_weekday) else str(best_matching_weekday).strip().lower()
    archetype_text = '' if pd.isna(daily_profile_archetype) else str(daily_profile_archetype).strip().lower()
    if best_text == 'saturday':
        return 'Saturday'
    if best_text == 'sunday':
        return 'Sunday'
    if 'saturday' in archetype_text:
        return 'Saturday-like'
    if 'sunday' in archetype_text:
        return 'Sunday-like'
    return 'Other / unclear'

if 'oos_point_error_df' not in globals() or oos_point_error_df.empty:
    oos_priority_rows = []
    for unique_id, series_run_map in rolling_runs.items():
        for target_date, run in series_run_map.items():
            if run is None or run.actual_profile is None:
                continue

            target_ts = pd.Timestamp(target_date).normalize()
            forecast_values = np.asarray(run.forecast_profile, dtype=float)
            actual_values = np.asarray(run.actual_profile, dtype=float)
            horizon = min(forecast_values.size, actual_values.size)
            recent_weekend_dates = getattr(run, 'recent_weekend_dates', []) or []
            recent_weekend_like = getattr(run, 'recent_weekend_like', None)
            recent_weekend_like = recent_weekend_like if recent_weekend_like is not None else 'None'

            for hour_idx in range(horizon):
                forecast_timestamp = run.forecast_start + pd.Timedelta(hours=hour_idx)
                hour_relative = hour_idx - int(run.forecast_start_offset_hours)
                actual_value = float(actual_values[hour_idx])
                forecast_value = float(forecast_values[hour_idx])
                signed_error = forecast_value - actual_value
                abs_error = abs(signed_error)
                ape_pct = np.nan
                if abs(actual_value) > 1e-9:
                    ape_pct = abs_error / abs(actual_value) * 100.0

                oos_priority_rows.append({
                    'unique_id': unique_id,
                    'target_date': target_ts.date().isoformat(),
                    'forecast_timestamp': forecast_timestamp,
                    'forecast_hour_of_day': int(forecast_timestamp.hour),
                    'window_slice': 'pre_holiday_14h' if hour_relative < 0 else 'holiday_24h',
                    'hour_relative_to_holiday': hour_relative,
                    'forecast_value': forecast_value,
                    'actual_value': actual_value,
                    'signed_error_oos': signed_error,
                    'abs_error_oos': abs_error,
                    'ape_pct_oos': ape_pct,
                    'recent_weekend_like_run': str(recent_weekend_like),
                    'recent_weekend_analogs_added': len(recent_weekend_dates),
                })

    if not oos_priority_rows:
        raise ValueError('No OOS forecast points with actuals were found in rolling_runs.')

    oos_point_error_df = pd.DataFrame(oos_priority_rows)
    oos_point_error_df = oos_point_error_df.merge(
        rolling_daily_table,
        on=['unique_id', 'target_date'],
        how='left',
        validate='many_to_one',
    )
    oos_point_error_df = oos_point_error_df.merge(
        selector_priority_df,
        on=['unique_id', 'target_date'],
        how='left',
        validate='many_to_one',
    )

oos_point_error_df['unique_id'] = oos_point_error_df['unique_id'].astype(str)
oos_point_error_df['weekend_like_category'] = oos_point_error_df.apply(_resolve_weekend_like_category_priority, axis=1)
oos_point_error_df['k_label'] = oos_point_error_df['k'].round().astype('Int64').astype(str)
oos_point_error_df['scale_method_label'] = oos_point_error_df['scale_method'].astype('string').fillna('None')
oos_point_error_df['selected_analogs_label'] = oos_point_error_df['selected_analogs'].round().astype('Int64').astype(str)
oos_point_error_df['eligible_tuning_dates_label'] = oos_point_error_df['eligible_tuning_dates'].round().astype('Int64').astype(str)
oos_point_error_df['hour_relative_to_holiday_label'] = oos_point_error_df['hour_relative_to_holiday'].astype('Int64').astype(str)
oos_point_error_df['recent_weekend_analogs_added_label'] = oos_point_error_df['recent_weekend_analogs_added'].astype('Int64').astype(str)

priority_segment_columns = [
    'unique_id',
    'selector_analog_cluster',
    'holiday_name',
    'anchor_holiday_name',
    'holiday_day_type',
    'weekday_name',
    'season',
    'date_rule',
    'best_matching_weekday',
    'weekend_like_category',
    'daily_profile_archetype',
    'event_profile_cluster',
    'typereg',
    'typedist',
    'k_label',
    'scale_method_label',
    'selected_analogs_label',
    'eligible_tuning_dates_label',
    'window_slice',
    'hour_relative_to_holiday_label',
    'recent_weekend_like_run',
    'recent_weekend_analogs_added_label',
]
priority_segment_columns = [
    column for column in priority_segment_columns if column in oos_point_error_df.columns
]

def build_high_error_low_sample_table(df, features, min_hour_samples=8, min_date_samples=2):
    summary_rows = []
    for feature in features:
        working_df = df[[
            feature,
            'target_date',
            'ape_pct_oos',
            'abs_error_oos',
            'selected_analogs',
            'eligible_tuning_dates',
            'recent_weekend_analogs_added',
        ]].copy()
        working_df = working_df.dropna(subset=[feature, 'ape_pct_oos', 'abs_error_oos'])
        if working_df.empty:
            continue

        feature_summary_df = (
            working_df.groupby(feature, dropna=False)
            .agg(
                hour_samples=('ape_pct_oos', 'size'),
                date_samples=('target_date', 'nunique'),
                mean_ape_pct=('ape_pct_oos', 'mean'),
                median_ape_pct=('ape_pct_oos', 'median'),
                q75_ape_pct=('ape_pct_oos', lambda values: values.quantile(0.75)),
                max_ape_pct=('ape_pct_oos', 'max'),
                mean_abs_error=('abs_error_oos', 'mean'),
                median_abs_error=('abs_error_oos', 'median'),
                median_selected_analogs=('selected_analogs', 'median'),
                median_eligible_tuning_dates=('eligible_tuning_dates', 'median'),
                median_recent_weekend_analogs_added=('recent_weekend_analogs_added', 'median'),
            )
            .reset_index()
        )
        feature_summary_df = feature_summary_df.loc[
            (feature_summary_df['hour_samples'] >= min_hour_samples)
            & (feature_summary_df['date_samples'] >= min_date_samples)
        ].copy()
        if feature_summary_df.empty:
            continue

        feature_summary_df = feature_summary_df.rename(columns={feature: 'segment_value'})
        feature_summary_df.insert(0, 'feature', feature)
        summary_rows.append(feature_summary_df)

    if not summary_rows:
        return pd.DataFrame()

    priority_df = pd.concat(summary_rows, ignore_index=True)
    priority_df['error_pressure_score'] = (
        0.6 * priority_df['median_ape_pct'].rank(pct=True)
        + 0.4 * priority_df['q75_ape_pct'].rank(pct=True)
    )
    priority_df['sample_scarcity_score'] = (
        0.6 * (1.0 - priority_df['hour_samples'].rank(pct=True))
        + 0.4 * (1.0 - priority_df['date_samples'].rank(pct=True))
    )
    priority_df['analog_scarcity_score'] = (
        0.5 * (1.0 - priority_df['median_selected_analogs'].rank(pct=True))
        + 0.5 * (1.0 - priority_df['median_eligible_tuning_dates'].rank(pct=True))
    )
    priority_df['priority_score'] = 100.0 * (
        0.55 * priority_df['error_pressure_score']
        + 0.30 * priority_df['sample_scarcity_score']
        + 0.15 * priority_df['analog_scarcity_score']
    )
    return priority_df.sort_values(
        ['priority_score', 'median_ape_pct', 'sample_scarcity_score'],
        ascending=[False, False, False],
    ).reset_index(drop=True)

def _recommend_priority_action(row):
    feature = str(row['feature'])
    median_selected_analogs = float(row['median_selected_analogs'])
    median_eligible_tuning_dates = float(row['median_eligible_tuning_dates'])
    weekend_analogs_added = float(row['median_recent_weekend_analogs_added'])

    if feature in {'selector_analog_cluster', 'event_profile_cluster', 'holiday_day_type', 'holiday_name', 'anchor_holiday_name'}:
        return 'Relajar filtro de cluster/subtipo o fusionar segmentos sparsos.'
    if feature in {'weekend_like_category', 'best_matching_weekday', 'daily_profile_archetype', 'recent_weekend_like_run'} and weekend_analogs_added < float(RECENT_WEEKEND_ANALOGS):
        return 'Agregar mas analogs recientes tipo sabado/domingo antes del ranking.'
    if feature in {'selected_analogs_label', 'eligible_tuning_dates_label', 'k_label'} or median_selected_analogs <= 3.5 or median_eligible_tuning_dates <= 4.5:
        return 'Ampliar el pool candidato antes del ranking o relajar filtros de seleccion.'
    if feature in {'window_slice', 'hour_relative_to_holiday_label'}:
        return 'Separar reglas para pre-holiday vs holiday o ajustar por bloque horario.'
    return 'Inspeccionar el segmento y considerar relajar filtros o sumar mas ejemplos recientes.'

def _priority_reason(row):
    return (
        f"APE mediana={row['median_ape_pct']:.2f}% | q75={row['q75_ape_pct']:.2f}% | "
        f"horas={int(row['hour_samples'])} | fechas={int(row['date_samples'])} | "
        f"analogs mediana={row['median_selected_analogs']:.1f} | elegibles medianos={row['median_eligible_tuning_dates']:.1f}"
    )

oos_action_priority_df = build_high_error_low_sample_table(
    oos_point_error_df,
    priority_segment_columns,
    min_hour_samples=8,
    min_date_samples=2,
 )

if oos_action_priority_df.empty:
    raise ValueError('No segment reached the minimum OOS sample thresholds for the priority table.')

oos_action_priority_df['priority_band'] = pd.cut(
    oos_action_priority_df['priority_score'],
    bins=[-np.inf, 45, 60, 75, np.inf],
    labels=['Monitor', 'Medium', 'High', 'Critical'],
)
oos_action_priority_df['recommended_action'] = oos_action_priority_df.apply(_recommend_priority_action, axis=1)
oos_action_priority_df['reason'] = oos_action_priority_df.apply(_priority_reason, axis=1)

critical_oos_action_priority_df = oos_action_priority_df.loc[
    oos_action_priority_df['priority_band'].isin(['High', 'Critical'])
] .copy()

print('Automatic OOS priority table | high error + low sample')
display(
    oos_action_priority_df[[
        'feature',
        'segment_value',
        'priority_band',
        'priority_score',
        'hour_samples',
        'date_samples',
        'median_ape_pct',
        'q75_ape_pct',
        'median_abs_error',
        'median_selected_analogs',
        'median_eligible_tuning_dates',
        'median_recent_weekend_analogs_added',
        'recommended_action',
        'reason',
    ]].head(40)
)

print('Critical / high-priority OOS segments to inspect first')
display(
    critical_oos_action_priority_df[[
        'feature',
        'segment_value',
        'priority_band',
        'priority_score',
        'hour_samples',
        'date_samples',
        'median_ape_pct',
        'q75_ape_pct',
        'median_abs_error',
        'recommended_action',
        'reason',
    ]].head(25)
)

Automatic OOS priority table | high error + low sample


,feature,segment_value,priority_band,priority_score,hour_samples,date_samples,median_ape_pct,q75_ape_pct,median_abs_error,median_selected_analogs,median_eligible_tuning_dates,median_recent_weekend_analogs_added,recommended_action,reason
0,weekend_like_category,Saturday-like,Critical,89.215517,114,3,8.922990,14.070918,1463.402362,2.0,19.0,0.0,Ampliar el pool candidato antes del ranking o ...,APE mediana=8.92% | q75=14.07% | horas=114 | f...
1,k_label,3,Critical,84.935345,190,3,10.624854,14.394006,195.424393,3.0,19.0,0.0,Ampliar el pool candidato antes del ranking o ...,APE mediana=10.62% | q75=14.39% | horas=190 | ...
2,selected_analogs_label,3,Critical,84.935345,190,3,10.624854,14.394006,195.424393,3.0,19.0,0.0,Ampliar el pool candidato antes del ranking o ...,APE mediana=10.62% | q75=14.39% | horas=190 | ...
3,selector_analog_cluster,O,Critical,82.480603,684,2,10.384957,15.272597,419.776482,4.5,19.0,0.0,Relajar filtro de cluster/subtipo o fusionar s...,APE mediana=10.38% | q75=15.27% | horas=684 | ...
4,holiday_name,New Year's Day,Critical,82.480603,684,2,10.384957,15.272597,419.776482,4.5,19.0,0.0,Relajar filtro de cluster/subtipo o fusionar s...,APE mediana=10.38% | q75=15.27% | horas=684 | ...
5,anchor_holiday_name,New Year's Day,Critical,82.480603,684,2,10.384957,15.272597,419.776482,4.5,19.0,0.0,Relajar filtro de cluster/subtipo o fusionar s...,APE mediana=10.38% | q75=15.27% | horas=684 | ...
6,typereg,PCR,Critical,81.849138,266,7,8.764579,14.771576,1365.614392,2.0,19.0,0.0,Ampliar el pool candidato antes del ranking o ...,APE mediana=8.76% | q75=14.77% | horas=266 | f...
7,daily_profile_archetype,Saturday-like,Critical,81.659483,266,7,8.764579,14.663226,1342.096835,2.0,19.0,0.0,Ampliar el pool candidato antes del ranking o ...,APE mediana=8.76% | q75=14.66% | horas=266 | f...
8,selector_analog_cluster,M,Critical,79.476293,684,2,9.143393,17.141281,482.885720,5.0,19.0,0.0,Relajar filtro de cluster/subtipo o fusionar s...,APE mediana=9.14% | q75=17.14% | horas=684 | f...
9,holiday_name,Martin Luther King Jr. Day,Critical,79.476293,684,2,9.143393,17.141281,482.885720,5.0,19.0,0.0,Relajar filtro de cluster/subtipo o fusionar s...,APE mediana=9.14% | q75=17.14% | horas=684 | f...


Critical / high-priority OOS segments to inspect first


,feature,segment_value,priority_band,priority_score,hour_samples,date_samples,median_ape_pct,q75_ape_pct,median_abs_error,recommended_action,reason
0,weekend_like_category,Saturday-like,Critical,89.215517,114,3,8.922990,14.070918,1463.402362,Ampliar el pool candidato antes del ranking o ...,APE mediana=8.92% | q75=14.07% | horas=114 | f...
1,k_label,3,Critical,84.935345,190,3,10.624854,14.394006,195.424393,Ampliar el pool candidato antes del ranking o ...,APE mediana=10.62% | q75=14.39% | horas=190 | ...
2,selected_analogs_label,3,Critical,84.935345,190,3,10.624854,14.394006,195.424393,Ampliar el pool candidato antes del ranking o ...,APE mediana=10.62% | q75=14.39% | horas=190 | ...
3,selector_analog_cluster,O,Critical,82.480603,684,2,10.384957,15.272597,419.776482,Relajar filtro de cluster/subtipo o fusionar s...,APE mediana=10.38% | q75=15.27% | horas=684 | ...
4,holiday_name,New Year's Day,Critical,82.480603,684,2,10.384957,15.272597,419.776482,Relajar filtro de cluster/subtipo o fusionar s...,APE mediana=10.38% | q75=15.27% | horas=684 | ...
5,anchor_holiday_name,New Year's Day,Critical,82.480603,684,2,10.384957,15.272597,419.776482,Relajar filtro de cluster/subtipo o fusionar s...,APE mediana=10.38% | q75=15.27% | horas=684 | ...
6,typereg,PCR,Critical,81.849138,266,7,8.764579,14.771576,1365.614392,Ampliar el pool candidato antes del ranking o ...,APE mediana=8.76% | q75=14.77% | horas=266 | f...
7,daily_profile_archetype,Saturday-like,Critical,81.659483,266,7,8.764579,14.663226,1342.096835,Ampliar el pool candidato antes del ranking o ...,APE mediana=8.76% | q75=14.66% | horas=266 | f...
8,selector_analog_cluster,M,Critical,79.476293,684,2,9.143393,17.141281,482.885720,Relajar filtro de cluster/subtipo o fusionar s...,APE mediana=9.14% | q75=17.14% | horas=684 | f...
9,holiday_name,Martin Luther King Jr. Day,Critical,79.476293,684,2,9.143393,17.141281,482.885720,Relajar filtro de cluster/subtipo o fusionar s...,APE mediana=9.14% | q75=17.14% | horas=684 | f...


In [11]:
import importlib.util

current_series_optuna_results = rolling_optuna_results.get(UNIQUE_ID, {}) if 'rolling_optuna_results' in globals() else {}
current_rolling_optuna_result = current_series_optuna_results.get(TARGET_DATE)
current_cluster_row = (
    rolling_daily_table.loc[
        (rolling_daily_table['unique_id'] == UNIQUE_ID)
        & (rolling_daily_table['target_date'] == TARGET_DATE)
    ]
    if 'rolling_daily_table' in globals() else pd.DataFrame()
)
current_cluster = current_cluster_row['analog_cluster'].iloc[0] if not current_cluster_row.empty else pd.NA
current_cluster_filter = (
    current_cluster_row['cluster_filter_label'].iloc[0]
    if (not current_cluster_row.empty and 'cluster_filter_label' in current_cluster_row.columns)
    else (True if MATCH_TARGET_CLUSTER else False)
)
current_regressor_params = (
    dict(current_rolling_optuna_result.best_config.get('regressor_params', {}))
    if current_rolling_optuna_result is not None else {}
)
current_k_range = (
    tuple(current_rolling_optuna_result.best_config.get('k_range', (np.nan, np.nan)))
    if current_rolling_optuna_result is not None else (np.nan, np.nan)
)
current_scale_method = (
    current_rolling_optuna_result.best_config.get('scale_method', SCALE_METHOD)
    if current_rolling_optuna_result is not None else SCALE_METHOD
)

optuna_typedist_choices = OPTUNA_TYPEDIST_CHOICES if OPTUNA_TYPEDIST_CHOICES is not None else ['pearson', 'euclidian']
optuna_typereg_choices = OPTUNA_TYPEREG_CHOICES if OPTUNA_TYPEREG_CHOICES is not None else ['PCR', 'PLS']
lgbm_available = importlib.util.find_spec('lightgbm') is not None

optuna_k_rule = f'integer in [{current_k_range[0]}, {current_k_range[1]}]'
optuna_scale_method_rule = OPTUNA_SCALE_METHOD_CHOICES if OPTUNA_SCALE_METHOD_CHOICES is not None else [SCALE_METHOD]
optuna_n_components_rule = 'integer in [2, min(k, season_length)] only when typereg is PCR or PLS'
optuna_lgbm_rule = (
    'available only via explicit typereg_choices override: n_estimators in {100, 200, 300}, '
    'learning_rate in {0.03, 0.05, 0.1}, num_leaves in {15, 31, 63}, min_child_samples in {10, 20, 30}'
)
optuna_runtime_note = (
    'DTW, RidgeReg, LassoReg, RF, OLSstep, and LGBM are excluded from the default Optuna grid, '
    'and k is capped by the realizable post-filter analog pool so Optuna does not request '
    'more neighbors than the workflow can actually use.'
)

if current_rolling_optuna_result is None:
    rolling_daily_table.loc[
        (rolling_daily_table['unique_id'] == UNIQUE_ID)
        & (rolling_daily_table['target_date'] == TARGET_DATE)
    ]
else:
    optuna_method_report = (
        f'OPTUNA METHOD REPORT FOR UNIQUE_ID={UNIQUE_ID} | TARGET_DATE={TARGET_DATE}\n'
        f'- Optimization mode: single-objective minimization.\n'
        f'- Sampler: TPESampler(seed={OPTUNA_RANDOM_SEED}).\n'
        f'- Search budget: n_trials={OPTUNA_N_TRIALS}, timeout_sec={OPTUNA_TIMEOUT_SEC}.\n'
        f'- Rolling cutoff: train_end={TARGET_DATE}; only dates strictly earlier than the target are used for tuning.\n'
        f'- Backtest folds: {len(current_rolling_optuna_result.eligible_dates)} eligible historical holiday dates, capped by OPTUNA_MAX_EVAL_DATES={OPTUNA_MAX_EVAL_DATES}.\n'
        f'- Cluster restriction: cluster={current_cluster_filter}; target analog_cluster={current_cluster if MATCH_TARGET_CLUSTER else "disabled"}.\n'
        f'- Objective function: minimize mean MAE across historical folds + 1000 * fail_rate.\n'
        f'- Search space for typedist: {optuna_typedist_choices}.\n'
        f'- Search space for typereg: {optuna_typereg_choices}.\n'
        f'- Search space for scale_method: {optuna_scale_method_rule}.\n'
        f'- Search space for k: {optuna_k_rule}.\n'
        f'- Conditional parameter for n_components: {optuna_n_components_rule}.\n'
        f'- Conditional LGBM parameters: '
        f'{optuna_lgbm_rule if lgbm_available else "not available because lightgbm is not installed"}.\n'
        f'- Runtime note: {optuna_runtime_note}\n'
        f'- Best configuration selected for this target: k(optuna)={current_rolling_optuna_result.best_config["k"]}, '
        f'typedist(optuna)={current_rolling_optuna_result.best_config["typedist"]}, '
        f'typereg(optuna)={current_rolling_optuna_result.best_config["typereg"]}, '
        f'scale_method(optuna)={current_scale_method}, '
        f'n_components(optuna)={current_rolling_optuna_result.best_config["n_components"]}, '
        f'regressor_params(optuna)={current_regressor_params}.\n'
        f'- k-range(optuna-param)=[{current_k_range[0]}, {current_k_range[1]}].'
    )
    print(optuna_method_report)
    display(current_rolling_optuna_result.summary_df)
    current_rolling_optuna_result.fold_metrics_df

OPTUNA METHOD REPORT FOR UNIQUE_ID=ERCOT_demand_ERCOT | TARGET_DATE=2026-03-02
- Optimization mode: single-objective minimization.
- Sampler: TPESampler(seed=42).
- Search budget: n_trials=25, timeout_sec=300.
- Rolling cutoff: train_end=2026-03-02; only dates strictly earlier than the target are used for tuning.
- Backtest folds: 19 eligible historical holiday dates, capped by OPTUNA_MAX_EVAL_DATES=19.
- Cluster restriction: cluster=holiday_identity; target analog_cluster=R.
- Objective function: minimize mean MAE across historical folds + 1000 * fail_rate.
- Search space for typedist: ['pearson', 'euclidian'].
- Search space for typereg: ['PCR', 'PLS', 'RidgeReg', 'LassoReg'].
- Search space for scale_method: [None, 'standard', 'minmax'].
- Search space for k: integer in [2, 7].
- Conditional parameter for n_components: integer in [2, min(k, season_length)] only when typereg is PCR or PLS.
- Conditional LGBM parameters: available only via explicit typereg_choices override: n_estimato

,param,value
0,cutoff_train,2026-03-02
1,eval_dates,19
2,FORECAST_START_OFFSET_HOURS,14
3,best_mean_mae,2821.257911
4,best_mean_mape_pct,5.21
5,best_fail_rate,0.0
6,OPTUNA_MIN_K,2
7,OPTUNA_MAX_K,7
8,SCALE_METHOD_CHOICES,"[None, standard, minmax]"
9,K,5


In [12]:
if (
    ('batch_result_2025_all' not in globals() or not batch_result_2025_all)
    and 'rolling_runs' in globals()
    and 'rolling_daily_table' in globals()
    and not rolling_daily_table.empty
    and 'rolling_target_items' in globals()
):
    batch_result_2025_all = {}
    for series_unique_id, series_run_map in rolling_runs.items():
        series_results_df = (
            rolling_daily_table
            .loc[rolling_daily_table['unique_id'] == series_unique_id]
            .reset_index(drop=True)
        )
        if series_results_df.empty:
            continue
        metric_summary_columns = [
            column for column in [
                'mae_holiday24_bias_adjusted',
                'mape_holiday24_bias_adjusted_pct',
                'mae_holiday24_raw',
                'mape_holiday24_raw_pct',
                'mae_window_bias_adjusted',
                'mape_window_bias_adjusted_pct',
                'mae_window',
                'mape_window_pct',
            ]
            if column in series_results_df.columns
        ]
        batch_result_2025_all[series_unique_id] = analog_holidays_module.AnalogHolidayBatchResult(
            target_items=rolling_target_items,
            runs=series_run_map,
            results_df=series_results_df,
            metric_summary_df=series_results_df[metric_summary_columns].describe(include='all'),
        )

batch_result_2025 = batch_result_2025_all.get(UNIQUE_ID) if 'batch_result_2025_all' in globals() else None
batch_results_to_plot = (
    batch_result_2025_all if 'batch_result_2025_all' in globals() and batch_result_2025_all else {}
)
if not batch_results_to_plot:
    raise ValueError('Run Cell 9 first to build rolling batch results before plotting.')

batch_inference_figures = {}
batch_inference_axes = {}

for series_unique_id, series_batch_result in batch_results_to_plot.items():
    print(f'Batch inference grid | {series_unique_id}')
    fig, axes = plot_batch_inference_grid(
        series_batch_result,
        title=(
            f'Batch inference | {series_unique_id}\n'
            f'Rolling nested tuning by target date | '
            f'window={SEASON_LENGTH}h | start=-{FORECAST_START_OFFSET_HOURS}h'
        ),
    )
    batch_inference_figures[series_unique_id] = fig
    batch_inference_axes[series_unique_id] = axes
    export_figure_pdf(fig, f'batch_inference_{series_unique_id}')
    plt.show()

Batch inference grid | ERCOT_demand_COAST
Saved PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/batch_inference_ERCOT_demand_COAST.pdf
Updated multipage PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/all_holiday_graphs_ercot.pdf
Batch inference grid | ERCOT_demand_EAST


/tmp/ipykernel_2652174/2106249098.py:60: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Saved PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/batch_inference_ERCOT_demand_EAST.pdf
Updated multipage PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/all_holiday_graphs_ercot.pdf
Batch inference grid | ERCOT_demand_ERCOT


/tmp/ipykernel_2652174/2106249098.py:60: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Saved PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/batch_inference_ERCOT_demand_ERCOT.pdf
Updated multipage PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/all_holiday_graphs_ercot.pdf
Batch inference grid | ERCOT_demand_FWEST


...

/tmp/ipykernel_2652174/2106249098.py:60: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Saved PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/batch_inference_ERCOT_demand_SOUTH.pdf
Updated multipage PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/all_holiday_graphs_ercot.pdf
Batch inference grid | ERCOT_demand_WEST


/tmp/ipykernel_2652174/2106249098.py:60: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Saved PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/batch_inference_ERCOT_demand_WEST.pdf
Updated multipage PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/all_holiday_graphs_ercot.pdf


/tmp/ipykernel_2652174/2106249098.py:60: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [13]:
current_series_optuna_results = rolling_optuna_results.get(UNIQUE_ID, {}) if 'rolling_optuna_results' in globals() else {}
current_rolling_optuna_result = current_series_optuna_results.get(TARGET_DATE)
resolved_regressor_params = dict(globals().get('REGRESSOR_PARAMS', {}))
resolved_scale_method = globals().get('SCALE_METHOD')
if current_rolling_optuna_result is not None:
    optuna_result = current_rolling_optuna_result
    K = int(optuna_result.best_config['k'])
    TYPEDIST = str(optuna_result.best_config['typedist'])
    TYPEREG = str(optuna_result.best_config['typereg'])
    SCALE_METHOD = optuna_result.best_config.get('scale_method', SCALE_METHOD)
    N_COMPONENTS = int(optuna_result.best_config['n_components'])
    resolved_regressor_params = dict(optuna_result.best_config.get('regressor_params', {}))
    resolved_scale_method = SCALE_METHOD
    REGRESSOR_PARAMS = resolved_regressor_params

SOURCE_PATH = _ensure_working_source_path(SOURCE_PATH)
run = batch_result_2025.runs.get(TARGET_DATE) if 'batch_result_2025' in globals() and batch_result_2025 is not None else None

if run is None:
    run = run_analog_holidays(
        unique_id=UNIQUE_ID,
        target_date=TARGET_DATE,
        source_path=SOURCE_PATH,
        season_length=SEASON_LENGTH,
        forecast_start_offset_hours=FORECAST_START_OFFSET_HOURS,
        k=K,
        typedist=TYPEDIST,
        typereg=TYPEREG,
        scale_method=resolved_scale_method,
        n_components=N_COMPONENTS,
        regressor_params=resolved_regressor_params,
        levels=LEVELS,
        special_labels=SPECIAL_LABELS,
        min_special_points=MIN_SPECIAL_POINTS,
        min_event_gap=MIN_EVENT_GAP,
        max_events=MAX_EVENTS,
        selector_features_path=SELECTOR_FEATURES_PATH,
        cluster_column=CLUSTER_COLUMN,
        match_target_cluster=MATCH_TARGET_CLUSTER,
        recent_weekend_analogs=RECENT_WEEKEND_ANALOGS,
    )

summary_df = build_run_summary(run)
print(summary_df.to_string(index=False))

                          metric              value
                       unique_id ERCOT_demand_ERCOT
                     target_date         2026-03-02
                  forecast_start   2026-03-01 10:00
                    forecast_end   2026-03-03 00:00
     forecast_start_offset_hours                 14
           forecast_window_hours                 38
                   target_exists               True
     target_has_complete_profile               True
                    target_label            holiday
             target_holiday_name               <NA>
                        typedist            pearson
                         typereg           RidgeReg
                    scale_method             minmax
                regressor_params                 {}
                     k_neighbors                  5
                      train_days               3337
              special_days_train                  9
             recent_weekend_like               None
recent_weeke

In [14]:
current_series_optuna_results = rolling_optuna_results.get(UNIQUE_ID, {}) if 'rolling_optuna_results' in globals() else {}
current_rolling_optuna_result = current_series_optuna_results.get(TARGET_DATE)
resolved_regressor_params = dict(globals().get('REGRESSOR_PARAMS', {}))
resolved_scale_method = globals().get('SCALE_METHOD')
if current_rolling_optuna_result is not None:
    optuna_result = current_rolling_optuna_result
    K = int(optuna_result.best_config['k'])
    TYPEDIST = str(optuna_result.best_config['typedist'])
    TYPEREG = str(optuna_result.best_config['typereg'])
    SCALE_METHOD = optuna_result.best_config.get('scale_method', SCALE_METHOD)
    N_COMPONENTS = int(optuna_result.best_config['n_components'])
    resolved_regressor_params = dict(optuna_result.best_config.get('regressor_params', {}))
    resolved_scale_method = SCALE_METHOD
    REGRESSOR_PARAMS = resolved_regressor_params

SOURCE_PATH = _ensure_working_source_path(SOURCE_PATH)
diagnostic_run = batch_result_2025.runs.get(TARGET_DATE) if 'batch_result_2025' in globals() and batch_result_2025 is not None else None

if diagnostic_run is None:
    diagnostic_run = run_analog_holidays(
        unique_id=UNIQUE_ID,
        target_date=TARGET_DATE,
        source_path=SOURCE_PATH,
        season_length=SEASON_LENGTH,
        forecast_start_offset_hours=FORECAST_START_OFFSET_HOURS,
        k=K,
        typedist=TYPEDIST,
        typereg=TYPEREG,
        scale_method=resolved_scale_method,
        n_components=N_COMPONENTS,
        regressor_params=resolved_regressor_params,
        levels=LEVELS,
        special_labels=SPECIAL_LABELS,
        min_special_points=MIN_SPECIAL_POINTS,
        min_event_gap=MIN_EVENT_GAP,
        max_events=MAX_EVENTS,
        expected_target_label=None,
        selector_features_path=SELECTOR_FEATURES_PATH,
        cluster_column=CLUSTER_COLUMN,
        match_target_cluster=MATCH_TARGET_CLUSTER,
        recent_weekend_analogs=RECENT_WEEKEND_ANALOGS,
    )

interval_rows = []
for lv in LEVELS:
    lo = diagnostic_run.interval_low.get(lv)
    hi = diagnostic_run.interval_high.get(lv)
    if lo is None or hi is None:
        continue
    width = hi - lo
    interval_rows.append({
        'level': lv,
        'average_interval_width': float(np.mean(width)),
        'max_interval_width': float(np.max(width)),
        'min_interval_width': float(np.min(width)),
    })

display(pd.DataFrame(interval_rows))

level_to_show = 95 if 95 in LEVELS else max(LEVELS)
hourly_interval_df = pd.DataFrame({
    'hour_relative_to_holiday': np.arange(len(diagnostic_run.forecast_profile)) - FORECAST_START_OFFSET_HOURS,
    'forecast_mean': diagnostic_run.forecast_profile,
    f'lower_limit_{level_to_show}': diagnostic_run.interval_low[level_to_show],
    f'upper_limit_{level_to_show}': diagnostic_run.interval_high[level_to_show],
})

hourly_interval_df.head(10)

,level,average_interval_width,max_interval_width,min_interval_width
0,50,1269.604156,2225.962041,268.710511
1,80,2398.090858,4146.737103,1040.403443
2,95,2962.334209,5236.620307,1426.249910


,hour_relative_to_holiday,forecast_mean,lower_limit_95,upper_limit_95
0,-14,48677.296550,47473.514476,49069.443048
1,-13,49454.808627,47714.753708,49535.405201
2,-12,50147.615528,48015.724400,50364.356110
3,-11,50623.641839,48384.093509,50993.235067
4,-10,51147.716078,48903.100936,51605.006374
5,-9,51305.819451,49191.402714,51885.944667
6,-8,51370.203844,49122.106630,52248.452229
7,-7,51327.466081,48731.397398,52498.034182
8,-6,50819.965950,48217.968146,52183.062125
9,-5,51037.081751,48616.549977,52179.502010


### X/X' and Y/Y' Sequence Batch Grid

One chart per forecast date, showing the historical X/X' pairs in light blue and the Y/Y' forecast sequence in red.

In this variant the X'/Y' window spans 38 hours and starts 14 hours before the holiday begins.

In [15]:
batch_results_to_plot = (
    batch_result_2025_all if 'batch_result_2025_all' in globals() and batch_result_2025_all else {UNIQUE_ID: batch_result_2025}
    if 'batch_result_2025' in globals() and batch_result_2025 is not None else {}
 )
if not batch_results_to_plot:
    raise ValueError('No batch results are available to plot.')

SOURCE_PATH = _ensure_working_source_path(SOURCE_PATH)
source_df_for_pair_grid = analog_holidays_module.load_audit_source(SOURCE_PATH)

batch_pair_figures = {}
batch_pair_axes = {}

for series_unique_id, series_batch_result in batch_results_to_plot.items():
    print(f"X/X' and Y/Y' grid | {series_unique_id}")
    adjusted_forecasts_by_date = None
    if 'bias_adjusted_profiles' in globals() and bias_adjusted_profiles:
        adjusted_forecasts_by_date = {
            target_date: bias_adjusted_profiles[(series_unique_id, target_date)]['adjusted_full_forecast']
            for target_date, _ in series_batch_result.target_items
            if (series_unique_id, target_date) in bias_adjusted_profiles
        }

    series_pair_df = (
        source_df_for_pair_grid.loc[source_df_for_pair_grid['unique_id'].astype(str) == str(series_unique_id)]
        .sort_values('date')
        .reset_index(drop=True)
    )
    post_holiday_actuals_by_date = {}
    if not series_pair_df.empty:
        for target_date, _ in series_batch_result.target_items:
            recovery_profile = analog_holidays_module._extract_hour_window(
                series_pair_df,
                window_start=pd.Timestamp(target_date).normalize() + pd.Timedelta(hours=24),
                length_hours=analog_holidays_module.POST_HOLIDAY_RECOVERY_HOURS,
            )
            if recovery_profile is not None:
                post_holiday_actuals_by_date[target_date] = recovery_profile

    fig_seq, axes_seq = plot_batch_pair_sequences_grid(
        series_batch_result,
        title=(
            f"X/X' and Y/Y' by forecast date | {series_unique_id}\n"
            f"Rolling nested tuning by target date | "
            f"window={SEASON_LENGTH}h | start=-{FORECAST_START_OFFSET_HOURS}h"
        ),
        adjusted_forecasts_by_date=adjusted_forecasts_by_date,
        post_holiday_actuals_by_date=post_holiday_actuals_by_date,
    )
    batch_pair_figures[series_unique_id] = fig_seq
    batch_pair_axes[series_unique_id] = axes_seq
    export_figure_pdf(fig_seq, f'batch_pair_sequences_{series_unique_id}')
    plt.show()

X/X' and Y/Y' grid | ERCOT_demand_COAST
Saved PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/batch_pair_sequences_ERCOT_demand_COAST.pdf
Updated multipage PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/all_holiday_graphs_ercot.pdf
X/X' and Y/Y' grid | ERCOT_demand_EAST


/tmp/ipykernel_2652174/3450347324.py:53: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Saved PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/batch_pair_sequences_ERCOT_demand_EAST.pdf
Updated multipage PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/all_holiday_graphs_ercot.pdf
X/X' and Y/Y' grid | ERCOT_demand_ERCOT


/tmp/ipykernel_2652174/3450347324.py:53: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Saved PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/batch_pair_sequences_ERCOT_demand_ERCOT.pdf
Updated multipage PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/all_holiday_graphs_ercot.pdf
X/X' and Y/Y' grid | ERCOT_demand_FWEST


...

/tmp/ipykernel_2652174/3450347324.py:53: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Saved PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/batch_pair_sequences_ERCOT_demand_SOUTH.pdf
Updated multipage PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/all_holiday_graphs_ercot.pdf
X/X' and Y/Y' grid | ERCOT_demand_WEST


/tmp/ipykernel_2652174/3450347324.py:53: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Saved PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/batch_pair_sequences_ERCOT_demand_WEST.pdf
Updated multipage PDF: /home/uriel/GIT/analog_holidays/Holiday_results_ercot/all_holiday_graphs_ercot.pdf


/tmp/ipykernel_2652174/3450347324.py:53: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### Register this run as a reproducible experiment

Saves all results of the current run into `experiments/experiment_<YYYY_MM_DD_HH_MM>/`
(`manifest.yaml` with the exact conditions, `metrics.csv`, `summary.csv`, `plots/`, `notes.md`)
for traceability and comparison with other experiments. See `experiments/README.md`.
Set `EXPERIMENT_SLUG = '...'` before running this cell to label the folder.

In [16]:
# === Register this run as a reproducible experiment under experiments/ ===
from analog_holidays.shared.experiment_logging import save_experiment_run

# Conditions actually used for this run (every mixed component). Optuna-tuned values vary per
# series; the per-series winners are kept inside metrics.csv.
EXPERIMENT_CONFIG = {
    'window': {
        'season_length': int(globals().get('SEASON_LENGTH', 38)),
        'forecast_start_offset_hours': int(globals().get('FORECAST_START_OFFSET_HOURS', 14)),
        'special_labels': list(globals().get('SPECIAL_LABELS', ('holiday',))),
    },
    'cluster': {
        'use_cluster': bool(globals().get('USE_CLUSTER', True)),
        'match_target_cluster': bool(globals().get('MATCH_TARGET_CLUSTER', True)),
        'cluster_column': globals().get('CLUSTER_COLUMN', 'analog_cluster'),
    },
    'analog_selection': {
        'typedist': globals().get('TYPEDIST'),
        'k': globals().get('K'),
        'min_special_points': globals().get('MIN_SPECIAL_POINTS'),
        'min_event_gap': globals().get('MIN_EVENT_GAP'),
        'max_events': globals().get('MAX_EVENTS'),
        'recent_weekend_analogs': globals().get('RECENT_WEEKEND_ANALOGS'),
    },
    'regression': {
        'typereg': globals().get('TYPEREG'),
        'scale_method': globals().get('SCALE_METHOD'),
        'n_components': globals().get('N_COMPONENTS'),
        'regressor_params': globals().get('REGRESSOR_PARAMS', {}),
        'levels': globals().get('LEVELS'),
    },
    'tuning': {
        'optuna_min_k': globals().get('OPTUNA_MIN_K'),
        'optuna_min_k_by_cluster': globals().get('OPTUNA_MIN_K_BY_CLUSTER'),
        'optuna_max_k_by_cluster': globals().get('OPTUNA_MAX_K_BY_CLUSTER'),
        'n_trials': globals().get('OPTUNA_N_TRIALS'),
        'timeout_sec': globals().get('OPTUNA_TIMEOUT_SEC'),
        'typedist_choices': globals().get('OPTUNA_TYPEDIST_CHOICES'),
        'typereg_choices': globals().get('OPTUNA_TYPEREG_CHOICES'),
        'scale_method_choices': globals().get('OPTUNA_SCALE_METHOD_CHOICES'),
    },
    'training_cutoff': 'rolling',
    'history_start': globals().get('HISTORY_START'),
    'source_path': str(globals().get('SOURCE_PATH', '')),
    'target_dates_var': 'TARGET_DATES_2025',
}

# Batch results to register: prefer the all-series dict, fall back to the single series.
_experiment_batch_results = (
    batch_result_2025_all
    if 'batch_result_2025_all' in globals() and batch_result_2025_all
    else ({UNIQUE_ID: batch_result_2025}
          if 'batch_result_2025' in globals() and batch_result_2025 is not None else {})
)

# Figures accumulated during the run (inference grids, pair-sequence grids, ...).
_experiment_figures = {}
_experiment_figures.update(globals().get('PDF_FIGURES', {}) or {})
# batch_pair_figures intentionally NOT merged: those grids are already in PDF_FIGURES under the
# descriptive 'batch_pair_sequences_<series>' key, so merging them would write a bare-name duplicate.

if _experiment_batch_results:
    EXPERIMENT_DIR = save_experiment_run(
        config=EXPERIMENT_CONFIG,
        batch_results=_experiment_batch_results,
        figures=_experiment_figures,
        selector_features_path=globals().get('SELECTOR_FEATURES_PATH'),
        slug=globals().get('EXPERIMENT_SLUG'),
    )
else:
    print('No batch results in memory -- run the batch cells before registering an experiment.')

Registered experiment: /home/uriel/GIT/analog_holidays/experiments/experiment_2026_06_25_13_07_holiday_identity_cluster_ercot
  metrics.csv rows : 171 (9 series)
  plots saved      : 20
